# Retinal DR DeepDRiD Smoke Test v0.1

**Project:** Cross-Modal Diagnostic Observability  
**Dataset:** DeepDRiD v1.1 regular fundus training and validation subsets  
**Purpose:** dataset audit, eye-identity resolution, image profiling, shortcut analysis, and eye-local diagnostic-observability testing.

This notebook was extracted from `Retinal_DR_Online_Smoke_Test_v0.1.ipynb`.

All DeepDRiD code cells and their existing outputs are preserved in their original order. The source archive and original image filenames are not modified.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

assert zip_path.exists(), zip_path

with zipfile.ZipFile(zip_path, "r") as z:
    license_files = [
        name for name in z.namelist()
        if "license" in Path(name).name.lower()
    ]

    print("LICENSE files found:")
    for name in license_files:
        print("-", name)

    for name in license_files:
        print("\n" + "=" * 100)
        print("FILE:", name)
        print("=" * 100)

        raw = z.read(name)

        for encoding in ["utf-8", "utf-8-sig", "gb18030", "latin-1"]:
            try:
                text = raw.decode(encoding)
                break
            except UnicodeDecodeError:
                continue

        print(text)

LICENSE files found:
- deepdrdoc-DeepDRiD-56d8af7/LICENSE

FILE: deepdrdoc-DeepDRiD-56d8af7/LICENSE
Attribution-ShareAlike 4.0 International


Creative Commons Corporation ("Creative Commons") is not a law firm and
does not provide legal services or legal advice. Distribution of
Creative Commons public licenses does not create a lawyer-client or
other relationship. Creative Commons makes its licenses and related
information available on an "as-is" basis. Creative Commons gives no
warranties regarding its licenses, any material licensed under their
terms and conditions, or any related information. Creative Commons
disclaims all liability for damages resulting from their use to the
fullest extent possible.

Using Creative Commons Public Licenses

Creative Commons public licenses provide a standard set of terms and
conditions that creators and other rights holders may use to share
original works of authorship and other material subject to copyright
and certain other rights specified in th

In [ ]:
from pathlib import Path
import requests
import subprocess

# DeepDRiD v1.1 的 Zenodo record
RECORD_ID = 8248825

# 直接保存到 Google Drive
SAVE_DIR = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 自动向 Zenodo 查询真正的文件名和下载地址
response = requests.get(
    f"https://zenodo.org/api/records/{RECORD_ID}",
    timeout=60
)
response.raise_for_status()
files = response.json().get("files", [])

matches = [
    f for f in files
    if f["key"].lower().endswith(".zip")
    and "v1.1" in f["key"].lower()
]

if len(matches) != 1:
    print("Zenodo 返回的文件：")
    for f in files:
        print(f["key"], f["size"])
    raise RuntimeError("没有唯一找到 DeepDRiD v1.1 ZIP，请停止。")

target = matches[0]
download_url = (
    target.get("links", {}).get("content")
    or target.get("links", {}).get("self")
)

filename = Path(target["key"]).name
expected_size = int(target["size"])

final_path = SAVE_DIR / filename
part_path = SAVE_DIR / f"{filename}.part"

print("Target:", filename)
print("Expected size:", round(expected_size / 1024**3, 3), "GB")
print("Saving to:", final_path)

if final_path.exists() and final_path.stat().st_size == expected_size:
    print("文件已经完整下载，无需重复下载。")
else:
    command = [
        "curl",
        "-L",
        "--fail",
        "--retry", "20",
        "--retry-delay", "5",
        "--continue-at", "-",
        "--output", str(part_path),
        download_url,
    ]

    result = subprocess.run(command)

    current_size = part_path.stat().st_size if part_path.exists() else 0
    print("\nCurrent size:", round(current_size / 1024**3, 3), "GB")

    if current_size == expected_size:
        part_path.replace(final_path)
        print("DOWNLOAD COMPLETE:", final_path)
    else:
        print("下载尚未完整。保留了 .part 文件；以后重新运行本代码即可断点续传。")

Target: DeepDRiD-v1.1.zip
Expected size: 1.279 GB
Saving to: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip

Current size: 1.279 GB
DOWNLOAD COMPLETE: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip


In [ ]:
from pathlib import Path
import hashlib
import requests

record_id = 8248825
zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

metadata = requests.get(
    f"https://zenodo.org/api/records/{record_id}",
    timeout=60
).json()

target = next(
    f for f in metadata["files"]
    if Path(f["key"]).name == zip_path.name
)

expected_md5 = target["checksum"].replace("md5:", "")

md5 = hashlib.md5()
with open(zip_path, "rb") as f:
    while True:
        chunk = f.read(8 * 1024 * 1024)
        if not chunk:
            break
        md5.update(chunk)

actual_md5 = md5.hexdigest()

print("Expected MD5:", expected_md5)
print("Actual MD5:  ", actual_md5)
print("CHECKSUM OK" if actual_md5 == expected_md5 else "CHECKSUM FAILED")

Expected MD5: 3379e2fd7a2dd398545a67148420a5d3
Actual MD5:   3379e2fd7a2dd398545a67148420a5d3
CHECKSUM OK


In [ ]:
from pathlib import Path
from collections import Counter
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

with zipfile.ZipFile(zip_path, "r") as z:
    infos = [x for x in z.infolist() if not x.is_dir()]

    total_size = sum(x.file_size for x in infos)
    level_counts = Counter()

    for x in infos:
        parts = Path(x.filename).parts
        key = "/".join(parts[:3])
        level_counts[key] += 1

    print("Total files:", len(infos))
    print("Uncompressed size:", round(total_size / 1024**3, 3), "GB")

    print("\nMain archive sections:")
    for name, count in level_counts.most_common(30):
        print(f"{count:6d}  {name}")

    print("\nFirst 40 file paths:")
    for x in infos[:40]:
        print(x.filename)

Total files: 2279
Uncompressed size: 1.324 GB

Main archive sections:
  1204  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-training
   406  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/Online-Challenge1&2-Evaluation
   404  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-validation
   156  deepdrdoc-DeepDRiD-56d8af7/ultra-widefield_images/ultra-widefield-training
    55  deepdrdoc-DeepDRiD-56d8af7/ultra-widefield_images/Online-Challenge3-Evaluation
    52  deepdrdoc-DeepDRiD-56d8af7/ultra-widefield_images/ultra-widefield-validation
     1  deepdrdoc-DeepDRiD-56d8af7/LICENSE
     1  deepdrdoc-DeepDRiD-56d8af7/README.md

First 40 file paths:
deepdrdoc-DeepDRiD-56d8af7/LICENSE
deepdrdoc-DeepDRiD-56d8af7/README.md
deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/Online-Challenge1&2-Evaluation/.~Readme.docx
deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/Online-Challenge1&2-Evaluation/Challenge1_labels.xlsx
deepdrdoc-DeepDRiD-56d8af7/regular_fundus_im

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    print("Non-image files in regular fundus training/validation:\n")

    for name in names:
        lower = name.lower()

        in_required_subset = (
            "regular_fundus_images/regular-fundus-training/" in name
            or "regular_fundus_images/regular-fundus-validation/" in name
        )

        is_image = lower.endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))

        if in_required_subset and not is_image and not name.endswith("/"):
            info = z.getinfo(name)
            print(f"{info.file_size:10d} bytes  {name}")

    print("\nREADME preview:\n")
    readme_name = next(
        name for name in names
        if name.endswith("/README.md")
    )

    text = z.read(readme_name).decode("utf-8", errors="replace")
    print(text[:6000])

Non-image files in regular fundus training/validation:

       162 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-training/.~Readme.docx
     17894 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-training/Readme.docx
     70299 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-training/regular-fundus-source-training.csv
     77040 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-training/regular-fundus-training.csv
       162 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-validation/.~Readme.docx
     17890 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-validation/Readme.docx
     24831 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-validation/regular-fundus-source-validation.csv
     27154 bytes  deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-validation/regular-fundus-validation.csv

README preview:

# Deep-D

In [ ]:
from pathlib import Path
from io import BytesIO
import zipfile
import pandas as pd

zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

wanted_suffixes = [
    "regular-fundus-training/regular-fundus-training.csv",
    "regular-fundus-training/regular-fundus-source-training.csv",
    "regular-fundus-validation/regular-fundus-validation.csv",
    "regular-fundus-validation/regular-fundus-source-validation.csv",
]

tables = {}

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    for suffix in wanted_suffixes:
        matches = [name for name in names if name.endswith(suffix)]

        if len(matches) != 1:
            print("ERROR:", suffix, "matches =", len(matches))
            continue

        internal_name = matches[0]
        raw = z.read(internal_name)

        try:
            df = pd.read_csv(BytesIO(raw), encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(BytesIO(raw), encoding="gbk")

        short_name = Path(internal_name).name
        tables[short_name] = df

        print("\n" + "=" * 100)
        print(short_name)
        print("Shape:", df.shape)
        print("Columns:", list(df.columns))
        print("\nFirst 8 rows:")
        print(df.head(8).to_string(index=False))

        print("\nLow-cardinality values:")
        for col in df.columns:
            n_unique = df[col].nunique(dropna=False)
            if n_unique <= 20:
                print(f"\n[{col}] unique={n_unique}")
                print(df[col].value_counts(dropna=False).head(20).to_string())


regular-fundus-training.csv
Shape: (1200, 10)
Columns: ['patient_id', 'image_id', 'image_path', 'Overall quality', 'left_eye_DR_Level', 'right_eye_DR_Level', 'patient_DR_Level', 'Clarity', 'Field definition', 'Artifact']

First 8 rows:
 patient_id image_id                          image_path  Overall quality  left_eye_DR_Level  right_eye_DR_Level  patient_DR_Level  Clarity  Field definition  Artifact
          1     1_l1 \regular-fundus-training\1\1_l1.jpg                0                0.0                 NaN                 0        8                 8         4
          1     1_l2 \regular-fundus-training\1\1_l2.jpg                0                0.0                 NaN                 0        8                 8         0
          1     1_r1 \regular-fundus-training\1\1_r1.jpg                0                NaN                 0.0                 0        8                 8         4
          1     1_r2 \regular-fundus-training\1\1_r2.jpg                0                Na

In [ ]:
import pandas as pd

train = tables["regular-fundus-training.csv"].copy()
train_source = tables["regular-fundus-source-training.csv"].copy()

val = tables["regular-fundus-validation.csv"].copy()
val_source = tables["regular-fundus-source-validation.csv"].copy()


def audit_split(name, labels, sources):
    print("\n" + "=" * 100)
    print(name)

    print("Rows:", len(labels))
    print("Patients:", labels["patient_id"].nunique())
    print("Unique image_id:", labels["image_id"].nunique())
    print("Duplicate image_id rows:", labels["image_id"].duplicated().sum())

    # 提取 l1、l2、r1、r2
    labels["view"] = labels["image_id"].str.extract(
        r"_(l1|l2|r1|r2)$",
        expand=False
    )

    print("\nViews per patient:")
    print(
        labels.groupby("patient_id")["image_id"]
        .count()
        .value_counts()
        .sort_index()
        .to_string()
    )

    expected_views = {"l1", "l2", "r1", "r2"}

    observed_views = (
        labels.groupby("patient_id")["view"]
        .apply(lambda x: set(x.dropna()))
    )

    bad_view_patients = observed_views[
        observed_views.apply(lambda x: x != expected_views)
    ]

    print("Patients without exactly l1/l2/r1/r2:",
          len(bad_view_patients))

    # 每位患者的 patient_DR_Level 是否唯一
    patient_label_counts = (
        labels.groupby("patient_id")["patient_DR_Level"]
        .nunique(dropna=False)
    )

    print("Patients with inconsistent patient_DR_Level:",
          int((patient_label_counts != 1).sum()))

    # 左眼 l1/l2 的眼级标签是否一致
    left = labels[labels["view"].isin(["l1", "l2"])]

    left_counts = (
        left.groupby("patient_id")["left_eye_DR_Level"]
        .nunique(dropna=False)
    )

    print("Patients with inconsistent left-eye label:",
          int((left_counts != 1).sum()))

    # 右眼 r1/r2 的眼级标签是否一致
    right = labels[labels["view"].isin(["r1", "r2"])]

    right_counts = (
        right.groupby("patient_id")["right_eye_DR_Level"]
        .nunique(dropna=False)
    )

    print("Patients with inconsistent right-eye label:",
          int((right_counts != 1).sum()))

    # 标签表与来源表是否一一匹配
    merged = labels.drop(columns=["view"]).merge(
        sources,
        on=["patient_id", "image_id", "image_path"],
        how="outer",
        indicator=True
    )

    print("\nLabel/source merge result:")
    print(merged["_merge"].value_counts().to_string())

    # 一个患者是否来自多个来源
    source_per_patient = (
        sources.groupby("patient_id")["Source"]
        .nunique(dropna=False)
    )

    print("Patients assigned to multiple sources:",
          int((source_per_patient > 1).sum()))

    print("\nPatients by source:")
    print(
        sources.drop_duplicates(["patient_id", "Source"])["Source"]
        .value_counts()
        .to_string()
    )


audit_split("TRAINING", train, train_source)
audit_split("VALIDATION", val, val_source)

train_patients = set(train["patient_id"])
val_patients = set(val["patient_id"])

train_images = set(train["image_id"])
val_images = set(val["image_id"])

print("\n" + "=" * 100)
print("CROSS-SPLIT LEAKAGE AUDIT")
print("Patient overlap:", len(train_patients & val_patients))
print("image_id overlap:", len(train_images & val_images))


TRAINING
Rows: 1200
Patients: 300
Unique image_id: 1200
Duplicate image_id rows: 0

Views per patient:
image_id
4    300
Patients without exactly l1/l2/r1/r2: 4
Patients with inconsistent patient_DR_Level: 0
Patients with inconsistent left-eye label: 0
Patients with inconsistent right-eye label: 0

Label/source merge result:
_merge
both          1200
left_only        0
right_only       0
Patients assigned to multiple sources: 0

Patients by source:
Source
Nicheng     285
Nation        9
Shanghai      6

VALIDATION
Rows: 400
Patients: 100
Unique image_id: 400
Duplicate image_id rows: 0

Views per patient:
image_id
4    100
Patients without exactly l1/l2/r1/r2: 0
Patients with inconsistent patient_DR_Level: 0
Patients with inconsistent left-eye label: 0
Patients with inconsistent right-eye label: 0

Label/source merge result:
_merge
both          400
left_only       0
right_only      0
Patients assigned to multiple sources: 0

Patients by source:
Source
Nicheng     92
Nation       5
Sha

In [ ]:
tmp = train.copy()

tmp["view_normalized"] = (
    tmp["image_id"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.extract(r"_(l1|l2|r1|r2)$", expand=False)
)

expected = {"l1", "l2", "r1", "r2"}

observed = (
    tmp.groupby("patient_id")["view_normalized"]
    .apply(lambda x: set(x.dropna()))
)

bad_patients = observed[
    observed.apply(lambda x: x != expected)
]

print("Bad patient count:", len(bad_patients))
print("\nObserved view sets:")
print(bad_patients.to_string())

for patient_id in bad_patients.index:
    print("\n" + "=" * 80)
    print("Patient:", patient_id)

    rows = tmp.loc[
        tmp["patient_id"] == patient_id,
        ["patient_id", "image_id", "image_path", "view_normalized"]
    ]

    for _, row in rows.iterrows():
        print(
            "image_id =", repr(row["image_id"]),
            "| path =", repr(row["image_path"]),
            "| normalized view =", repr(row["view_normalized"])
        )

Bad patient count: 4

Observed view sets:
patient_id
56     {l1, r2}
77           {}
164        {r1}
167    {l2, r2}

Patient: 56
image_id = '56_l1' | path = '\\regular-fundus-training\\56\\56_l1.jpg' | normalized view = 'l1'
image_id = '56_r3' | path = '\\regular-fundus-training\\56\\56_r3.jpg' | normalized view = nan
image_id = '56_l3' | path = '\\regular-fundus-training\\56\\56_l3.jpg' | normalized view = nan
image_id = '56_r2' | path = '\\regular-fundus-training\\56\\56_r2.jpg' | normalized view = 'r2'

Patient: 77
image_id = '77_r4' | path = '\\regular-fundus-training\\77\\77_r4.jpg' | normalized view = nan
image_id = '77_r3' | path = '\\regular-fundus-training\\77\\77_r3.jpg' | normalized view = nan
image_id = '77_l3' | path = '\\regular-fundus-training\\77\\77_l3.jpg' | normalized view = nan
image_id = '77_l4' | path = '\\regular-fundus-training\\77\\77_l4.jpg' | normalized view = nan

Patient: 164
image_id = '164_r3' | path = '\\regular-fundus-training\\164\\164_r3.jpg' | norma

In [ ]:
import pandas as pd

def audit_general_views(name, df):
    x = df.copy()

    parsed = x["image_id"].astype(str).str.strip().str.lower().str.extract(
        r"_(l|r)([1-4])$"
    )
    x["eye"] = parsed[0]
    x["view_number"] = parsed[1]

    print("\n" + "=" * 90)
    print(name)

    print("Unparsed image IDs:", int(x["eye"].isna().sum()))

    print("\nSuffix distribution:")
    suffix = x["eye"].fillna("?") + x["view_number"].fillna("?")
    print(suffix.value_counts().sort_index().to_string())

    counts = (
        x.groupby(["patient_id", "eye"])
        .size()
        .unstack(fill_value=0)
    )

    print("\nImages per eye pattern:")
    print(counts.value_counts().to_string())

    bad_counts = counts[
        (counts.get("l", 0) != 2) |
        (counts.get("r", 0) != 2)
    ]

    print("\nPatients not having exactly 2 left + 2 right images:",
          len(bad_counts))

    duplicate_eye_views = (
        x.dropna(subset=["eye", "view_number"])
        .duplicated(["patient_id", "eye", "view_number"])
        .sum()
    )

    print("Duplicate view number within the same patient and eye:",
          int(duplicate_eye_views))


audit_general_views("TRAINING", train)
audit_general_views("VALIDATION", val)


TRAINING
Unparsed image IDs: 0

Suffix distribution:
l1    297
l2    297
l3      4
l4      1
r1    297
r2    298
r3      4
r4      2

Images per eye pattern:
l  r
2  2    299
1  3      1

Patients not having exactly 2 left + 2 right images: 1
Duplicate view number within the same patient and eye: 0

VALIDATION
Unparsed image IDs: 0

Suffix distribution:
l1    100
l2    100
r1    100
r2    100

Images per eye pattern:
l  r
2  2    100

Patients not having exactly 2 left + 2 right images: 0
Duplicate view number within the same patient and eye: 0


In [ ]:
cols = [
    "patient_id", "image_id", "image_path",
    "left_eye_DR_Level", "right_eye_DR_Level",
    "patient_DR_Level", "Overall quality",
    "Clarity", "Field definition", "Artifact"
]

print(
    train.loc[train["patient_id"] == 164, cols]
    .to_string(index=False)
)

print("\nSource:")
print(
    train_source.loc[
        train_source["patient_id"] == 164,
        ["patient_id", "image_id", "Source"]
    ].to_string(index=False)
)

 patient_id image_id                              image_path  left_eye_DR_Level  right_eye_DR_Level  patient_DR_Level  Overall quality  Clarity  Field definition  Artifact
        164   164_r3 \regular-fundus-training\164\164_r3.jpg                0.0                 NaN                 0                0        8                 4         0
        164   164_r4 \regular-fundus-training\164\164_r4.jpg                0.0                 NaN                 0                0       10                 4         0
        164   164_r1 \regular-fundus-training\164\164_r1.jpg                NaN                 0.0                 0                1        8                 8         4
        164   164_l3 \regular-fundus-training\164\164_l3.jpg                NaN                 0.0                 0                1        8                10         1

Source:
 patient_id image_id  Source
        164   164_r3 Nicheng
        164   164_r4 Nicheng
        164   164_r1 Nicheng
        164   1

In [ ]:
import numpy as np
import pandas as pd

def audit_eye_name_consistency(name, df):
    x = df.copy()

    x["eye_from_filename"] = (
        x["image_id"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.extract(r"_(l|r)[1-4]$", expand=False)
    )

    left_present = x["left_eye_DR_Level"].notna()
    right_present = x["right_eye_DR_Level"].notna()

    x["eye_from_label"] = np.select(
        [
            left_present & ~right_present,
            right_present & ~left_present
        ],
        [
            "l",
            "r"
        ],
        default="ambiguous"
    )

    mismatches = x[
        x["eye_from_filename"] != x["eye_from_label"]
    ]

    print("\n" + "=" * 90)
    print(name)
    print("Rows:", len(x))
    print("Ambiguous label-eye rows:",
          int((x["eye_from_label"] == "ambiguous").sum()))
    print("Filename/label eye mismatches:", len(mismatches))

    if len(mismatches):
        print(
            mismatches[
                [
                    "patient_id",
                    "image_id",
                    "eye_from_filename",
                    "eye_from_label",
                    "left_eye_DR_Level",
                    "right_eye_DR_Level"
                ]
            ].to_string(index=False)
        )

audit_eye_name_consistency("TRAINING", train)
audit_eye_name_consistency("VALIDATION", val)


TRAINING
Rows: 1200
Ambiguous label-eye rows: 0
Filename/label eye mismatches: 11
 patient_id image_id eye_from_filename eye_from_label  left_eye_DR_Level  right_eye_DR_Level
         56    56_r3                 r              l                1.0                 NaN
         56    56_l3                 l              r                NaN                 0.0
         77    77_r4                 r              l                0.0                 NaN
         77    77_r3                 r              l                0.0                 NaN
         77    77_l3                 l              r                NaN                 0.0
         77    77_l4                 l              r                NaN                 0.0
        164   164_r3                 r              l                0.0                 NaN
        164   164_r4                 r              l                0.0                 NaN
        164   164_l3                 l              r                NaN        

In [ ]:
import numpy as np
import pandas as pd

def audit_label_derived_eye(name, df):
    x = df.copy()

    x["eye_from_filename"] = (
        x["image_id"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.extract(r"_(l|r)[1-4]$", expand=False)
    )

    left_present = x["left_eye_DR_Level"].notna()
    right_present = x["right_eye_DR_Level"].notna()

    x["eye_from_label"] = np.select(
        [
            left_present & ~right_present,
            right_present & ~left_present
        ],
        ["l", "r"],
        default="ambiguous"
    )

    x["eye_conflict"] = (
        x["eye_from_filename"] != x["eye_from_label"]
    )

    counts = (
        x.groupby(["patient_id", "eye_from_label"])
        .size()
        .unstack(fill_value=0)
    )

    print("\n" + "=" * 90)
    print(name)
    print("Filename/label conflicts:", int(x["eye_conflict"].sum()))
    print("Ambiguous rows:", int((x["eye_from_label"] == "ambiguous").sum()))

    print("\nImages per patient using label-derived eye:")
    print(counts.value_counts().to_string())

    bad = counts[
        (counts.get("l", 0) != 2) |
        (counts.get("r", 0) != 2)
    ]

    print("\nPatients not restored to exactly 2 left + 2 right:",
          len(bad))

    conflicts = x.loc[
        x["eye_conflict"],
        [
            "patient_id",
            "image_id",
            "eye_from_filename",
            "eye_from_label",
            "left_eye_DR_Level",
            "right_eye_DR_Level"
        ]
    ].copy()

    return x, conflicts


train_checked, train_conflicts = audit_label_derived_eye(
    "TRAINING", train
)

val_checked, val_conflicts = audit_label_derived_eye(
    "VALIDATION", val
)

print("\nConflict patients:")
print(sorted(train_conflicts["patient_id"].unique().tolist()))


TRAINING
Filename/label conflicts: 11
Ambiguous rows: 0

Images per patient using label-derived eye:
l  r
2  2    300

Patients not restored to exactly 2 left + 2 right: 0

VALIDATION
Filename/label conflicts: 0
Ambiguous rows: 0

Images per patient using label-derived eye:
l  r
2  2    100

Patients not restored to exactly 2 left + 2 right: 0

Conflict patients:
[56, 77, 164, 167]


In [ ]:
from pathlib import Path
from io import BytesIO
import zipfile
import xml.etree.ElementTree as ET

zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

with zipfile.ZipFile(zip_path, "r") as outer_zip:
    readme_matches = [
        name for name in outer_zip.namelist()
        if name.endswith(
            "regular_fundus_images/"
            "regular-fundus-training/Readme.docx"
        )
    ]

    if len(readme_matches) != 1:
        raise RuntimeError(
            f"Expected one training Readme.docx, found {len(readme_matches)}"
        )

    docx_bytes = outer_zip.read(readme_matches[0])

with zipfile.ZipFile(BytesIO(docx_bytes), "r") as docx_zip:
    xml_bytes = docx_zip.read("word/document.xml")

root = ET.fromstring(xml_bytes)

ns = {
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
}

paragraphs = []

for paragraph in root.findall(".//w:p", ns):
    texts = [
        node.text
        for node in paragraph.findall(".//w:t", ns)
        if node.text
    ]

    line = "".join(texts).strip()

    if line:
        paragraphs.append(line)

print("\n".join(paragraphs))

Dear friends. This regular-fundus-training.7z file contains two types of files: fundus images and label CSV file.
There are 1200 regular fundus images from 300 patients for training.
CSV file label:
Num
Label
Level
1
patient_id
Patients with the serial number.
2
image_id
Image sequence number.
3
image_path
Local path of fundus image.
4
Overall quality
0
Quality is not good enough for the diagnosis of retinal diseases
5
1
Quality is good enough for the diagnosis of retinal diseases
6
Artifact
0
Do not contain artifacts
7
1
Outside the aortic arch with range less than 1/4 of the image
8
4
Do not affect the macular area with scope less than 1/4
9
6
Cover more than 1/4, less than 1/2 of the image
10
8
Cover more than 1/2 without fully cover the posterior pole
11
10
Cover the entire posterior pole
12
Clarity
1
Only Level 1 vascular arch can be identified
13
4
Can identify Level 2 vascular arch and a small number of lesions
14
6
Can identify Level 3 vascular arch and some lesions
15
8
Can id

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD-v1.1.zip"
)

extract_root = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD_v1.1_Extracted"
)

wanted_prefixes = [
    "deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-training/",
    "deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images/regular-fundus-validation/",
    "deepdrdoc-DeepDRiD-56d8af7/LICENSE",
    "deepdrdoc-DeepDRiD-56d8af7/README.md",
]

extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    selected = [
        name for name in z.namelist()
        if any(name.startswith(prefix) for prefix in wanted_prefixes)
    ]

    print("Files selected:", len(selected))

    for i, name in enumerate(selected, start=1):
        z.extract(name, extract_root)

        if i % 200 == 0 or i == len(selected):
            print(f"Extracted {i}/{len(selected)}")

print("EXTRACTION COMPLETE")
print("Saved to:", extract_root)

Files selected: 2014
Extracted 200/2014
Extracted 400/2014
Extracted 600/2014
Extracted 800/2014
Extracted 1000/2014
Extracted 1200/2014
Extracted 1400/2014
Extracted 1600/2014
Extracted 1800/2014
Extracted 2000/2014
Extracted 2014/2014
EXTRACTION COMPLETE
Saved to: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD_v1.1_Extracted


In [ ]:
from pathlib import Path
import pandas as pd

base = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/DeepDRiD_Official_Raw/DeepDRiD_v1.1_Extracted/"
    "deepdrdoc-DeepDRiD-56d8af7/regular_fundus_images"
)

for folder_name, expected_images in [
    ("regular-fundus-training", 1200),
    ("regular-fundus-validation", 400),
]:
    split_dir = base / folder_name
    image_dir = split_dir / "Images"
    label_csv = split_dir / f"{folder_name}.csv"
    source_csv = split_dir / f"regular-fundus-source-{folder_name.split('-')[-1]}.csv"

    images = sorted(image_dir.rglob("*.jpg"))
    labels = pd.read_csv(label_csv, encoding="utf-8-sig")
    sources = pd.read_csv(source_csv, encoding="utf-8-sig")

    image_stems = [p.stem for p in images]
    disk_ids = set(image_stems)
    label_ids = set(labels["image_id"].astype(str))
    source_ids = set(sources["image_id"].astype(str))

    print("\n" + "=" * 90)
    print(folder_name)
    print("Expected images:", expected_images)
    print("Images on disk:", len(images))
    print("Label rows:", len(labels))
    print("Source rows:", len(sources))
    print("Duplicate image filenames:", len(image_stems) - len(disk_ids))
    print("Labels missing images:", len(label_ids - disk_ids))
    print("Images missing labels:", len(disk_ids - label_ids))
    print("Source IDs missing labels:", len(source_ids - label_ids))
    print("Label IDs missing source:", len(label_ids - source_ids))

    if (
        len(images) == expected_images
        and len(labels) == expected_images
        and len(sources) == expected_images
        and disk_ids == label_ids == source_ids
    ):
        print("EXTRACTION AND LINKAGE OK")
    else:
        print("CHECK REQUIRED")


regular-fundus-training
Expected images: 1200
Images on disk: 1200
Label rows: 1200
Source rows: 1200
Duplicate image filenames: 0
Labels missing images: 0
Images missing labels: 0
Source IDs missing labels: 0
Label IDs missing source: 0
EXTRACTION AND LINKAGE OK

regular-fundus-validation
Expected images: 400
Images on disk: 400
Label rows: 400
Source rows: 400
Duplicate image filenames: 0
Labels missing images: 0
Images missing labels: 0
Source IDs missing labels: 0
Label IDs missing source: 0
EXTRACTION AND LINKAGE OK


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

# ---------- 1. 合并标签与来源 ----------
def build_canonical_metadata(split_name, labels, sources, split_folder):
    x = labels.merge(
        sources,
        on=["patient_id", "image_id", "image_path"],
        how="inner",
        validate="one_to_one"
    ).copy()

    x["split"] = split_name

    # 从原始文件名读取眼别和视图编号
    parsed = (
        x["image_id"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.extract(r"_(l|r)([1-4])$")
    )

    x["eye_from_filename"] = parsed[0]
    x["view_number"] = pd.to_numeric(parsed[1], errors="coerce")

    # 依据非空的左右眼诊断标签确定真实眼别
    left_present = x["left_eye_DR_Level"].notna()
    right_present = x["right_eye_DR_Level"].notna()

    x["eye_resolved"] = np.select(
        [
            left_present & ~right_present,
            right_present & ~left_present
        ],
        ["l", "r"],
        default="ambiguous"
    )

    x["eye_name_conflict"] = (
        x["eye_from_filename"] != x["eye_resolved"]
    )

    # 统一生成眼级 DR 标签
    x["eye_DR_Level"] = np.where(
        x["eye_resolved"] == "l",
        x["left_eye_DR_Level"],
        x["right_eye_DR_Level"]
    )

    # 对应 Drive 中的实际图像路径
    x["image_path_disk"] = x.apply(
        lambda row: str(
            base
            / split_folder
            / "Images"
            / str(row["patient_id"])
            / f"{row['image_id']}.jpg"
        ),
        axis=1
    )

    x["image_exists"] = x["image_path_disk"].map(
        lambda p: Path(p).exists()
    )

    return x


train_canonical = build_canonical_metadata(
    "training",
    train,
    train_source,
    "regular-fundus-training"
)

val_canonical = build_canonical_metadata(
    "validation",
    val,
    val_source,
    "regular-fundus-validation"
)

canonical = pd.concat(
    [train_canonical, val_canonical],
    ignore_index=True
)

conflicts = canonical.loc[
    canonical["eye_name_conflict"]
].copy()


# ---------- 2. 最终安全检查 ----------
assert len(canonical) == 1600
assert canonical["image_id"].is_unique
assert canonical["image_exists"].all()
assert (canonical["eye_resolved"] != "ambiguous").all()

assert canonical.loc[
    canonical["split"] == "training", "patient_id"
].nunique() == 300

assert canonical.loc[
    canonical["split"] == "validation", "patient_id"
].nunique() == 100

assert len(conflicts) == 11


# ---------- 3. 保存到独立结果目录 ----------
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

output_dir = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "06_Data_Records/DeepDRiD/Smoke_Test_v0.1"
) / run_id

output_dir.mkdir(parents=True, exist_ok=True)

canonical_path = output_dir / "deepdrid_canonical_metadata.csv"
conflict_path = output_dir / "eye_name_conflicts.csv"
summary_path = output_dir / "audit_summary.json"

canonical.to_csv(canonical_path, index=False)
conflicts.to_csv(conflict_path, index=False)

summary = {
    "dataset": "DeepDRiD",
    "version": "v1.1",
    "zenodo_record": "8248825",
    "total_images": 1600,
    "training_images": 1200,
    "validation_images": 400,
    "training_patients": 300,
    "validation_patients": 100,
    "patient_overlap": 0,
    "image_id_overlap": 0,
    "missing_images_after_extraction": int(
        (~canonical["image_exists"]).sum()
    ),
    "eye_name_conflicts": int(
        canonical["eye_name_conflict"].sum()
    ),
    "conflict_patients": sorted(
        conflicts["patient_id"].unique().astype(int).tolist()
    ),
    "eye_resolution_rule": (
        "Resolve eye identity from the non-null left/right eye DR label; "
        "preserve original image_id and filename."
    ),
    "quality_definition": {
        "Overall quality 0": "Not sufficient for retinal diagnosis",
        "Overall quality 1": "Sufficient for retinal diagnosis",
        "Clarity": "Higher is better",
        "Field definition": "Higher is better",
        "Artifact": "Higher is worse"
    }
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved to:", output_dir)
print("Canonical rows:", len(canonical))
print("Images present:", int(canonical["image_exists"].sum()))
print("Eye-name conflicts:", len(conflicts))
print("Conflict patients:",
      sorted(conflicts["patient_id"].unique().tolist()))

print("\nFiles:")
for p in sorted(output_dir.iterdir()):
    print("-", p.name)

Saved to: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/DeepDRiD/Smoke_Test_v0.1/20260718T220050Z
Canonical rows: 1600
Images present: 1600
Eye-name conflicts: 11
Conflict patients: [56, 77, 164, 167]

Files:
- audit_summary.json
- deepdrid_canonical_metadata.csv
- eye_name_conflicts.csv


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from PIL import Image

profile_rows = []

for i, row in canonical.iterrows():
    image_path = Path(row["image_path_disk"])

    with Image.open(image_path) as img:
        img = img.convert("RGB")

        original_width, original_height = img.size
        file_size = image_path.stat().st_size

        # 统一缩小后计算指标，减少运行时间
        img_small = img.copy()
        img_small.thumbnail((768, 768))

        rgb = np.asarray(img_small, dtype=np.uint8)

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    red = rgb[:, :, 0].astype(np.float32)
    green = rgb[:, :, 1].astype(np.float32)
    blue = rgb[:, :, 2].astype(np.float32)

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    black_mask = np.all(rgb < 10, axis=2)

    profile_rows.append({
        "split": row["split"],
        "patient_id": row["patient_id"],
        "image_id": row["image_id"],
        "eye_resolved": row["eye_resolved"],
        "eye_DR_Level": row["eye_DR_Level"],
        "patient_DR_Level": row["patient_DR_Level"],
        "Overall quality": row["Overall quality"],
        "Clarity": row["Clarity"],
        "Field definition": row["Field definition"],
        "Artifact": row["Artifact"],
        "Source": row["Source"],

        "width": original_width,
        "height": original_height,
        "aspect_ratio": original_width / original_height,
        "file_size_bytes": file_size,

        "gray_mean": float(gray.mean()),
        "gray_std": float(gray.std()),
        "black_border_ratio": float(black_mask.mean()),
        "laplacian_variance": float(
            cv2.Laplacian(gray, cv2.CV_64F).var()
        ),

        "red_mean": float(red.mean()),
        "green_mean": float(green.mean()),
        "blue_mean": float(blue.mean()),

        "saturation_mean": float(hsv[:, :, 1].mean()),
        "value_mean": float(hsv[:, :, 2].mean()),
    })

    if (i + 1) % 100 == 0 or (i + 1) == len(canonical):
        print(f"Profiled {i + 1}/{len(canonical)}")


deepdrid_profile = pd.DataFrame(profile_rows)

profile_path = output_dir / "deepdrid_image_profile.csv"
deepdrid_profile.to_csv(profile_path, index=False)

print("\nPROFILE COMPLETE")
print("Rows:", len(deepdrid_profile))
print("Patients:", deepdrid_profile["patient_id"].nunique())
print("Saved to:", profile_path)

print("\nMetric summary:")
print(
    deepdrid_profile[
        [
            "width",
            "height",
            "gray_mean",
            "gray_std",
            "black_border_ratio",
            "laplacian_variance",
            "saturation_mean"
        ]
    ].describe().to_string()
)

Profiled 100/1600
Profiled 200/1600
Profiled 300/1600
Profiled 400/1600
Profiled 500/1600
Profiled 600/1600
Profiled 700/1600
Profiled 800/1600
Profiled 900/1600
Profiled 1000/1600
Profiled 1100/1600
Profiled 1200/1600
Profiled 1300/1600
Profiled 1400/1600
Profiled 1500/1600
Profiled 1600/1600

PROFILE COMPLETE
Rows: 1600
Patients: 400
Saved to: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/DeepDRiD/Smoke_Test_v0.1/20260718T220050Z/deepdrid_image_profile.csv

Metric summary:
             width       height    gray_mean     gray_std  black_border_ratio  laplacian_variance  saturation_mean
count  1600.000000  1600.000000  1600.000000  1600.000000         1600.000000         1600.000000      1600.000000
mean   1821.007500  1875.305000    67.533791    53.429011            0.322866           28.424736       115.327705
std     122.089091    77.218626    23.962795    13.157781            0.039262           26.379486        22.543536
min    1592.000000  1725.00000

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
)

df = deepdrid_profile.copy()

# 眼级 DR >= 2 定义为 referable DR
df["referable_DR"] = (
    pd.to_numeric(df["eye_DR_Level"], errors="coerce") >= 2
).astype(int)

train_df = df[df["split"] == "training"].copy()
val_df = df[df["split"] == "validation"].copy()


# 纯基础质量指标
quality_features = [
    "gray_mean",
    "gray_std",
    "black_border_ratio",
    "laplacian_variance",
    "saturation_mean",
    "value_mean",
]

# 加入采集/文件属性后的较强简单基线
quality_acquisition_features = quality_features + [
    "width",
    "height",
    "aspect_ratio",
    "file_size_bytes",
    "red_mean",
    "green_mean",
    "blue_mean",
]

feature_sets = {
    "quality_only": quality_features,
    "quality_plus_acquisition": quality_acquisition_features,
}

targets = {
    "official_quality": "Overall quality",
    "referable_DR": "referable_DR",
}

result_rows = []
prediction_rows = []

for target_name, target_col in targets.items():
    y_train = train_df[target_col].astype(int)
    y_val = val_df[target_col].astype(int)

    print("\n" + "=" * 100)
    print("TARGET:", target_name)
    print("Training prevalence:", round(y_train.mean(), 4))
    print("Validation prevalence:", round(y_val.mean(), 4))

    for feature_set_name, features in feature_sets.items():
        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )
            ),
        ])

        model.fit(train_df[features], y_train)

        probabilities = model.predict_proba(
            val_df[features]
        )[:, 1]

        predictions = (probabilities >= 0.5).astype(int)

        roc_auc = roc_auc_score(
            y_val,
            probabilities
        )

        average_precision = average_precision_score(
            y_val,
            probabilities
        )

        balanced_accuracy = balanced_accuracy_score(
            y_val,
            predictions
        )

        result_rows.append({
            "target": target_name,
            "feature_set": feature_set_name,
            "n_train": len(train_df),
            "n_validation": len(val_df),
            "validation_prevalence": float(y_val.mean()),
            "roc_auc": float(roc_auc),
            "average_precision": float(average_precision),
            "balanced_accuracy_at_0.5": float(
                balanced_accuracy
            ),
        })

        for image_id, patient_id, truth, prob in zip(
            val_df["image_id"],
            val_df["patient_id"],
            y_val,
            probabilities,
        ):
            prediction_rows.append({
                "target": target_name,
                "feature_set": feature_set_name,
                "patient_id": patient_id,
                "image_id": image_id,
                "truth": int(truth),
                "probability": float(prob),
            })

        print("\nFeature set:", feature_set_name)
        print("ROC AUC:", round(roc_auc, 4))
        print("Average precision:", round(average_precision, 4))
        print(
            "Balanced accuracy:",
            round(balanced_accuracy, 4)
        )


# 找出验证集中表现最强的单一普通指标
print("\n" + "=" * 100)
print("BEST SINGLE-METRIC RESULTS")

single_metric_rows = []

for target_name, target_col in targets.items():
    y_val = val_df[target_col].astype(int)

    for feature in quality_acquisition_features:
        values = val_df[feature]

        if values.nunique(dropna=True) < 2:
            continue

        valid = values.notna() & y_val.notna()

        raw_auc = roc_auc_score(
            y_val[valid],
            values[valid]
        )

        oriented_auc = max(
            raw_auc,
            1.0 - raw_auc
        )

        single_metric_rows.append({
            "target": target_name,
            "feature": feature,
            "raw_auc": float(raw_auc),
            "orientation_free_auc": float(
                oriented_auc
            ),
        })

single_metric_results = pd.DataFrame(
    single_metric_rows
)

for target_name in targets:
    top = (
        single_metric_results[
            single_metric_results["target"] == target_name
        ]
        .sort_values(
            "orientation_free_auc",
            ascending=False
        )
        .head(5)
    )

    print("\nTarget:", target_name)
    print(top.to_string(index=False))


# 保存结果
baseline_results = pd.DataFrame(result_rows)
validation_predictions = pd.DataFrame(
    prediction_rows
)

baseline_path = (
    output_dir /
    "simple_quality_baseline_results.csv"
)

prediction_path = (
    output_dir /
    "simple_quality_validation_predictions.csv"
)

single_metric_path = (
    output_dir /
    "single_metric_validation_auc.csv"
)

baseline_results.to_csv(
    baseline_path,
    index=False
)

validation_predictions.to_csv(
    prediction_path,
    index=False
)

single_metric_results.to_csv(
    single_metric_path,
    index=False
)

print("\n" + "=" * 100)
print("SUMMARY")
print(
    baseline_results.to_string(index=False)
)

print("\nSaved:")
print("-", baseline_path.name)
print("-", prediction_path.name)
print("-", single_metric_path.name)


TARGET: official_quality
Training prevalence: 0.48
Validation prevalence: 0.455

Feature set: quality_only
ROC AUC: 0.7186
Average precision: 0.6161
Balanced accuracy: 0.6787

Feature set: quality_plus_acquisition
ROC AUC: 0.7801
Average precision: 0.6806
Balanced accuracy: 0.7287

TARGET: referable_DR
Training prevalence: 0.4333
Validation prevalence: 0.45

Feature set: quality_only
ROC AUC: 0.795
Average precision: 0.6987
Balanced accuracy: 0.7472

Feature set: quality_plus_acquisition
ROC AUC: 0.8124
Average precision: 0.7588
Balanced accuracy: 0.7664

BEST SINGLE-METRIC RESULTS

Target: official_quality
          target            feature  raw_auc  orientation_free_auc
official_quality    saturation_mean 0.623500              0.623500
official_quality    file_size_bytes 0.616620              0.616620
official_quality laplacian_variance 0.585064              0.585064
official_quality          blue_mean 0.426076              0.573924
official_quality       aspect_ratio 0.573495     

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


# ============================================================
# 1. 将两张同眼图像聚合为一个 eye-level 样本
# ============================================================

df = deepdrid_profile.copy()

df["referable_DR"] = (
    pd.to_numeric(df["eye_DR_Level"], errors="coerce") >= 2
).astype(int)

metric_columns = [
    "width",
    "height",
    "aspect_ratio",
    "file_size_bytes",
    "gray_mean",
    "gray_std",
    "black_border_ratio",
    "laplacian_variance",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
]

eye_rows = []

for keys, group in df.groupby(
    ["split", "patient_id", "eye_resolved", "Source"],
    dropna=False
):
    split, patient_id, eye_resolved, source = keys

    quality_values = group["Overall quality"].astype(int)

    if quality_values.min() == 1:
        quality_pattern = "both_good"
    elif quality_values.max() == 0:
        quality_pattern = "both_poor"
    else:
        quality_pattern = "mixed"

    row = {
        "split": split,
        "patient_id": patient_id,
        "eye_resolved": eye_resolved,
        "Source": source,
        "n_views": len(group),
        "eye_DR_Level": group["eye_DR_Level"].iloc[0],
        "referable_DR": group["referable_DR"].iloc[0],
        "quality_pattern": quality_pattern,
    }

    for metric in metric_columns:
        row[metric] = pd.to_numeric(
            group[metric],
            errors="coerce"
        ).mean()

    eye_rows.append(row)

eye_df = pd.DataFrame(eye_rows)

print("Eye-level rows:", len(eye_df))
print("Training eyes:",
      len(eye_df[eye_df["split"] == "training"]))
print("Validation eyes:",
      len(eye_df[eye_df["split"] == "validation"]))

print("\nViews per eye:")
print(eye_df["n_views"].value_counts().sort_index().to_string())

print("\nQuality patterns:")
print(
    eye_df.groupby(["split", "quality_pattern"])
    .size()
    .to_string()
)

assert len(eye_df) == 800
assert (eye_df["n_views"] == 2).all()


# ============================================================
# 2. 三类特征，帮助区分混杂来源
# ============================================================

feature_sets = {
    "technical_geometry": [
        "width",
        "height",
        "aspect_ratio",
        "file_size_bytes",
        "black_border_ratio",
        "laplacian_variance",
    ],

    "photometric": [
        "gray_mean",
        "gray_std",
        "red_mean",
        "green_mean",
        "blue_mean",
        "saturation_mean",
        "value_mean",
    ],

    "all_simple_metrics": metric_columns,
}


# ============================================================
# 3. 定义四种审计条件
# ============================================================

subgroups = {
    "all_eyes": lambda x: pd.Series(
        True,
        index=x.index
    ),

    "Nicheng_only": lambda x: (
        x["Source"] == "Nicheng"
    ),

    "both_good_quality": lambda x: (
        x["quality_pattern"] == "both_good"
    ),

    "both_poor_quality": lambda x: (
        x["quality_pattern"] == "both_poor"
    ),
}


# ============================================================
# 4. 在相同子群内训练和独立验证
# ============================================================

result_rows = []

for subgroup_name, subgroup_function in subgroups.items():

    train_eye = eye_df[
        (eye_df["split"] == "training")
        & subgroup_function(eye_df)
    ].copy()

    val_eye = eye_df[
        (eye_df["split"] == "validation")
        & subgroup_function(eye_df)
    ].copy()

    y_train = train_eye["referable_DR"].astype(int)
    y_val = val_eye["referable_DR"].astype(int)

    print("\n" + "=" * 100)
    print("SUBGROUP:", subgroup_name)
    print("Training eyes:", len(train_eye))
    print("Validation eyes:", len(val_eye))
    print("Training prevalence:",
          round(y_train.mean(), 4) if len(y_train) else "NA")
    print("Validation prevalence:",
          round(y_val.mean(), 4) if len(y_val) else "NA")

    if (
        len(train_eye) < 20
        or len(val_eye) < 20
        or y_train.nunique() < 2
        or y_val.nunique() < 2
    ):
        print("Skipped: insufficient samples or only one class.")
        continue

    for feature_set_name, features in feature_sets.items():

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )
            ),
        ])

        model.fit(
            train_eye[features],
            y_train
        )

        probability = model.predict_proba(
            val_eye[features]
        )[:, 1]

        auc = roc_auc_score(
            y_val,
            probability
        )

        ap = average_precision_score(
            y_val,
            probability
        )

        result_rows.append({
            "subgroup": subgroup_name,
            "feature_set": feature_set_name,
            "n_train_eyes": len(train_eye),
            "n_validation_eyes": len(val_eye),
            "validation_prevalence": float(y_val.mean()),
            "roc_auc": float(auc),
            "average_precision": float(ap),
        })

        print(
            feature_set_name,
            "| AUC =", round(auc, 4),
            "| AP =", round(ap, 4)
        )


# ============================================================
# 5. 保存并显示结果
# ============================================================

confounding_audit = pd.DataFrame(result_rows)

save_path = (
    output_dir /
    "eye_level_quality_source_confounding_audit.csv"
)

confounding_audit.to_csv(
    save_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL TABLE")
print(
    confounding_audit.sort_values(
        ["subgroup", "feature_set"]
    ).to_string(index=False)
)

print("\nSaved to:", save_path)

Eye-level rows: 800
Training eyes: 600
Validation eyes: 200

Views per eye:
n_views
2    800

Quality patterns:
split       quality_pattern
training    both_good          288
            both_poor          312
validation  both_good           91
            both_poor          109

SUBGROUP: all_eyes
Training eyes: 600
Validation eyes: 200
Training prevalence: 0.4333
Validation prevalence: 0.45
technical_geometry | AUC = 0.7756 | AP = 0.7351
photometric | AUC = 0.8036 | AP = 0.7561
all_simple_metrics | AUC = 0.8084 | AP = 0.7537

SUBGROUP: Nicheng_only
Training eyes: 570
Validation eyes: 184
Training prevalence: 0.4263
Validation prevalence: 0.4457
technical_geometry | AUC = 0.7581 | AP = 0.712
photometric | AUC = 0.8044 | AP = 0.7573
all_simple_metrics | AUC = 0.8025 | AP = 0.759

SUBGROUP: both_good_quality
Training eyes: 288
Validation eyes: 91
Training prevalence: 0.4653
Validation prevalence: 0.4835
technical_geometry | AUC = 0.7669 | AP = 0.7615
photometric | AUC = 0.8017 | AP = 0.

In [ ]:
import pandas as pd
import numpy as np

audit = deepdrid_profile.copy()

audit["eye_DR_Level"] = pd.to_numeric(
    audit["eye_DR_Level"],
    errors="coerce"
).astype(int)

audit["referable_DR"] = (
    audit["eye_DR_Level"] >= 2
).astype(int)

audit["resolution"] = (
    audit["width"].astype(int).astype(str)
    + "x"
    + audit["height"].astype(int).astype(str)
)

print("=" * 100)
print("RESOLUTION INVENTORY")

print("\nUnique resolutions by split:")
print(
    audit.groupby("split")["resolution"]
    .nunique()
    .to_string()
)

print("\nUnique resolutions by split and source:")
print(
    audit.groupby(["split", "Source"])["resolution"]
    .nunique()
    .to_string()
)

train_resolutions = set(
    audit.loc[
        audit["split"] == "training",
        "resolution"
    ]
)

val_resolutions = set(
    audit.loc[
        audit["split"] == "validation",
        "resolution"
    ]
)

print("\nResolution overlap:")
print("Training-only:", sorted(train_resolutions - val_resolutions))
print("Validation-only:", sorted(val_resolutions - train_resolutions))
print("Shared:", sorted(train_resolutions & val_resolutions))


for split_name in ["training", "validation"]:

    sub = audit[
        (audit["split"] == split_name)
        & (audit["Source"] == "Nicheng")
    ].copy()

    print("\n" + "=" * 100)
    print("NICHENG:", split_name.upper())
    print("Images:", len(sub))
    print("Patients:", sub["patient_id"].nunique())

    resolution_stats = (
        sub.groupby("resolution")
        .agg(
            images=("image_id", "size"),
            patients=("patient_id", "nunique"),
            referable_rate=("referable_DR", "mean"),
            quality_good_rate=("Overall quality", "mean"),
            median_black_border=(
                "black_border_ratio",
                "median"
            ),
            median_gray=("gray_mean", "median"),
            median_file_size=(
                "file_size_bytes",
                "median"
            ),
        )
        .sort_values("images", ascending=False)
    )

    print("\nResolution groups:")
    print(
        resolution_stats
        .head(15)
        .round(4)
        .to_string()
    )

    top_resolutions = (
        sub["resolution"]
        .value_counts()
        .head(10)
        .index
    )

    top_sub = sub[
        sub["resolution"].isin(top_resolutions)
    ]

    grade_table = pd.crosstab(
        top_sub["resolution"],
        top_sub["eye_DR_Level"]
    )

    grade_table["Total"] = grade_table.sum(axis=1)

    grade_table = grade_table.sort_values(
        "Total",
        ascending=False
    )

    print("\nDR-grade counts for the most common resolutions:")
    print(grade_table.to_string())

    rate_table = pd.crosstab(
        top_sub["resolution"],
        top_sub["eye_DR_Level"],
        normalize="index"
    )

    print("\nDR-grade proportions within resolution:")
    print(
        rate_table
        .round(3)
        .to_string()
    )


print("\n" + "=" * 100)
print("GRADE-WISE ACQUISITION STATISTICS — NICHENG ONLY")

nicheng = audit[
    audit["Source"] == "Nicheng"
].copy()

grade_stats = (
    nicheng.groupby(["split", "eye_DR_Level"])
    .agg(
        images=("image_id", "size"),
        patients=("patient_id", "nunique"),
        median_width=("width", "median"),
        median_height=("height", "median"),
        median_black_border=(
            "black_border_ratio",
            "median"
        ),
        median_gray=("gray_mean", "median"),
        median_red=("red_mean", "median"),
        median_file_size=(
            "file_size_bytes",
            "median"
        ),
        median_sharpness=(
            "laplacian_variance",
            "median"
        ),
    )
)

print(
    grade_stats
    .round(4)
    .to_string()
)

RESOLUTION INVENTORY

Unique resolutions by split:
split
training      6
validation    3

Unique resolutions by split and source:
split       Source  
training    Nation      4
            Nicheng     6
            Shanghai    2
validation  Nation      3
            Nicheng     3
            Shanghai    2

Resolution overlap:
Training-only: ['1592x1728', '2230x1725', '2232x1727']
Validation-only: []
Shared: ['1734x1821', '1736x1824', '1976x1984']

NICHENG: TRAINING
Images: 1140
Patients: 285

Resolution groups:
            images  patients  referable_rate  quality_good_rate  median_black_border  median_gray  median_file_size
resolution                                                                                                         
1736x1824      720       183          0.2778             0.4194               0.3164      70.2241          522684.5
1976x1984      388        97          0.7113             0.5722               0.3469      56.1520          566475.0
1734x1821       12 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


# ============================================================
# 1. 重新构建带固定分辨率信息的眼级样本
# ============================================================

img = deepdrid_profile.copy()

img["referable_DR"] = (
    pd.to_numeric(
        img["eye_DR_Level"],
        errors="coerce"
    ) >= 2
).astype(int)

img["resolution"] = (
    img["width"].astype(int).astype(str)
    + "x"
    + img["height"].astype(int).astype(str)
)

features_all = [
    "gray_mean",
    "gray_std",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
    "black_border_ratio",
    "laplacian_variance",
    "file_size_bytes",
]

eye_rows = []

for keys, group in img.groupby(
    ["split", "patient_id", "eye_resolved"],
    dropna=False
):
    split, patient_id, eye = keys

    resolutions = sorted(
        group["resolution"].dropna().unique().tolist()
    )

    row = {
        "split": split,
        "patient_id": patient_id,
        "eye_resolved": eye,
        "Source": group["Source"].iloc[0],
        "referable_DR": int(
            group["referable_DR"].iloc[0]
        ),
        "eye_DR_Level": int(
            group["eye_DR_Level"].iloc[0]
        ),
        "n_views": len(group),
        "n_resolutions": len(resolutions),
        "resolution": (
            resolutions[0]
            if len(resolutions) == 1
            else "mixed"
        ),
    }

    for feature in features_all:
        row[feature] = pd.to_numeric(
            group[feature],
            errors="coerce"
        ).mean()

    eye_rows.append(row)

fixed_eye_df = pd.DataFrame(eye_rows)

print("Eye-level rows:", len(fixed_eye_df))
print(
    "Eyes containing more than one resolution:",
    int((fixed_eye_df["n_resolutions"] != 1).sum())
)

print("\nEye counts by split, source and resolution:")
print(
    fixed_eye_df.groupby(
        ["split", "Source", "resolution"]
    )
    .size()
    .to_string()
)


# ============================================================
# 2. 在 Nicheng 内分别锁死两个主要分辨率
# ============================================================

feature_sets = {
    "strict_photometric": [
        "gray_mean",
        "gray_std",
        "red_mean",
        "green_mean",
        "blue_mean",
        "saturation_mean",
        "value_mean",
    ],

    "border_file_sharpness": [
        "black_border_ratio",
        "file_size_bytes",
        "laplacian_variance",
    ],

    "all_without_dimensions": features_all,
}

fixed_resolutions = [
    "1736x1824",
    "1976x1984",
]

result_rows = []

for resolution in fixed_resolutions:

    train_fixed = fixed_eye_df[
        (fixed_eye_df["split"] == "training")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    val_fixed = fixed_eye_df[
        (fixed_eye_df["split"] == "validation")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    y_train = train_fixed["referable_DR"].astype(int)
    y_val = val_fixed["referable_DR"].astype(int)

    print("\n" + "=" * 100)
    print("FIXED RESOLUTION:", resolution)
    print("Training eyes:", len(train_fixed))
    print("Validation eyes:", len(val_fixed))
    print(
        "Training prevalence:",
        round(y_train.mean(), 4)
    )
    print(
        "Validation prevalence:",
        round(y_val.mean(), 4)
    )

    for feature_set_name, features in feature_sets.items():

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )
            ),
        ])

        model.fit(
            train_fixed[features],
            y_train
        )

        probability = model.predict_proba(
            val_fixed[features]
        )[:, 1]

        auc = roc_auc_score(
            y_val,
            probability
        )

        ap = average_precision_score(
            y_val,
            probability
        )

        result_rows.append({
            "source": "Nicheng",
            "resolution": resolution,
            "feature_set": feature_set_name,
            "n_train_eyes": len(train_fixed),
            "n_validation_eyes": len(val_fixed),
            "training_prevalence": float(
                y_train.mean()
            ),
            "validation_prevalence": float(
                y_val.mean()
            ),
            "roc_auc": float(auc),
            "average_precision": float(ap),
        })

        print(
            feature_set_name,
            "| AUC =", round(auc, 4),
            "| AP =", round(ap, 4)
        )


# ============================================================
# 3. 保存
# ============================================================

fixed_resolution_results = pd.DataFrame(
    result_rows
)

save_path = (
    output_dir /
    "fixed_resolution_DR_shortcut_audit.csv"
)

fixed_resolution_results.to_csv(
    save_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL TABLE")
print(
    fixed_resolution_results
    .sort_values(
        ["resolution", "feature_set"]
    )
    .to_string(index=False)
)

print("\nSaved to:", save_path)

Eye-level rows: 800
Eyes containing more than one resolution: 26

Eye counts by split, source and resolution:
split       Source    resolution
training    Nation    1736x1824       8
                      1976x1984       8
                      mixed           2
            Nicheng   1592x1728       2
                      1736x1824     354
                      1976x1984     194
                      mixed          20
            Shanghai  1736x1824       4
                      1976x1984       8
validation  Nation    1736x1824       3
                      1976x1984       6
                      mixed           1
            Nicheng   1736x1824     135
                      1976x1984      46
                      mixed           3
            Shanghai  1736x1824       4
                      1976x1984       2

FIXED RESOLUTION: 1736x1824
Training eyes: 354
Validation eyes: 135
Training prevalence: 0.2797
Validation prevalence: 0.3407
strict_photometric | AUC = 0.7616 | AP = 0.5882
bo

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest


photometric_features = [
    "gray_mean",
    "gray_std",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
]

pair_detail_rows = []
pair_summary_rows = []

for resolution in ["1736x1824", "1976x1984"]:

    # --------------------------------------------------------
    # 1. 在相同来源、相同分辨率的训练眼上训练
    # --------------------------------------------------------
    train_sub = fixed_eye_df[
        (fixed_eye_df["split"] == "training")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    val_sub = fixed_eye_df[
        (fixed_eye_df["split"] == "validation")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42,
            )
        ),
    ])

    model.fit(
        train_sub[photometric_features],
        train_sub["referable_DR"].astype(int)
    )

    val_sub["probability"] = model.predict_proba(
        val_sub[photometric_features]
    )[:, 1]

    # --------------------------------------------------------
    # 2. 仅保留同一患者两只眼都在该分辨率内的患者
    # --------------------------------------------------------
    complete_patients = (
        val_sub.groupby("patient_id")
        .filter(
            lambda g:
                len(g) == 2
                and g["eye_resolved"].nunique() == 2
        )
        .copy()
    )

    total_complete_patients = (
        complete_patients["patient_id"].nunique()
    )

    # 一只眼阴性、一只眼阳性
    discordant = (
        complete_patients.groupby("patient_id")
        .filter(
            lambda g:
                set(g["referable_DR"].astype(int)) == {0, 1}
        )
        .copy()
    )

    discordant_patient_ids = sorted(
        discordant["patient_id"].unique().tolist()
    )

    correct_count = 0
    score_differences = []

    for patient_id in discordant_patient_ids:

        g = discordant[
            discordant["patient_id"] == patient_id
        ].copy()

        positive = g[
            g["referable_DR"] == 1
        ].iloc[0]

        negative = g[
            g["referable_DR"] == 0
        ].iloc[0]

        difference = (
            positive["probability"]
            - negative["probability"]
        )

        is_correct = difference > 0

        correct_count += int(is_correct)
        score_differences.append(float(difference))

        pair_detail_rows.append({
            "resolution": resolution,
            "patient_id": patient_id,
            "referable_eye": positive["eye_resolved"],
            "nonreferable_eye": negative["eye_resolved"],
            "referable_grade": int(
                positive["eye_DR_Level"]
            ),
            "nonreferable_grade": int(
                negative["eye_DR_Level"]
            ),
            "referable_probability": float(
                positive["probability"]
            ),
            "nonreferable_probability": float(
                negative["probability"]
            ),
            "probability_difference": float(
                difference
            ),
            "correct_pair_ranking": bool(
                is_correct
            ),
        })

    n_pairs = len(discordant_patient_ids)

    if n_pairs > 0:
        pair_accuracy = correct_count / n_pairs

        p_value = binomtest(
            correct_count,
            n_pairs,
            p=0.5,
            alternative="greater"
        ).pvalue

        # 患者对 bootstrap 置信区间
        rng = np.random.default_rng(42)
        differences = np.asarray(score_differences)

        bootstrap_accuracy = []

        for _ in range(10000):
            sample = rng.choice(
                differences,
                size=len(differences),
                replace=True
            )

            bootstrap_accuracy.append(
                np.mean(sample > 0)
            )

        ci_low, ci_high = np.quantile(
            bootstrap_accuracy,
            [0.025, 0.975]
        )

        mean_difference = float(
            differences.mean()
        )
    else:
        pair_accuracy = np.nan
        p_value = np.nan
        ci_low = np.nan
        ci_high = np.nan
        mean_difference = np.nan

    pair_summary_rows.append({
        "source": "Nicheng",
        "resolution": resolution,
        "complete_two_eye_validation_patients":
            total_complete_patients,
        "discordant_validation_pairs": n_pairs,
        "correct_pair_rankings": correct_count,
        "pairwise_accuracy": pair_accuracy,
        "bootstrap_CI_low": ci_low,
        "bootstrap_CI_high": ci_high,
        "exact_binomial_p": p_value,
        "mean_probability_difference":
            mean_difference,
    })

    print("\n" + "=" * 100)
    print("RESOLUTION:", resolution)
    print(
        "Complete two-eye validation patients:",
        total_complete_patients
    )
    print(
        "Discordant fellow-eye pairs:",
        n_pairs
    )
    print(
        "Correct pair rankings:",
        correct_count
    )
    print(
        "Pairwise accuracy:",
        round(pair_accuracy, 4)
        if n_pairs else "NA"
    )
    print(
        "95% bootstrap CI:",
        (
            round(ci_low, 4),
            round(ci_high, 4)
        )
        if n_pairs else "NA"
    )
    print(
        "Exact binomial p:",
        round(p_value, 6)
        if n_pairs else "NA"
    )
    print(
        "Mean probability difference:",
        round(mean_difference, 4)
        if n_pairs else "NA"
    )


pair_summary = pd.DataFrame(pair_summary_rows)
pair_details = pd.DataFrame(pair_detail_rows)

summary_path = (
    output_dir /
    "discordant_fellow_eye_pair_summary.csv"
)

details_path = (
    output_dir /
    "discordant_fellow_eye_pair_details.csv"
)

pair_summary.to_csv(
    summary_path,
    index=False
)

pair_details.to_csv(
    details_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL PAIRED SUMMARY")
print(pair_summary.to_string(index=False))

print("\nSaved:")
print("-", summary_path.name)
print("-", details_path.name)


RESOLUTION: 1736x1824
Complete two-eye validation patients: 66
Discordant fellow-eye pairs: 6
Correct pair rankings: 4
Pairwise accuracy: 0.6667
95% bootstrap CI: (np.float64(0.3333), np.float64(1.0))
Exact binomial p: 0.34375
Mean probability difference: 0.0052

RESOLUTION: 1976x1984
Complete two-eye validation patients: 23
Discordant fellow-eye pairs: 4
Correct pair rankings: 1
Pairwise accuracy: 0.25
95% bootstrap CI: (np.float64(0.0), np.float64(0.75))
Exact binomial p: 0.9375
Mean probability difference: -0.0593

FINAL PAIRED SUMMARY
 source resolution  complete_two_eye_validation_patients  discordant_validation_pairs  correct_pair_rankings  pairwise_accuracy  bootstrap_CI_low  bootstrap_CI_high  exact_binomial_p  mean_probability_difference
Nicheng  1736x1824                                    66                            6                      4           0.666667          0.333333               1.00           0.34375                     0.005209
Nicheng  1976x1984            

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import binomtest


photometric_features = [
    "gray_mean",
    "gray_std",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
]

oof_detail_rows = []
oof_summary_rows = []

for resolution in ["1736x1824", "1976x1984"]:

    data = fixed_eye_df[
        (fixed_eye_df["split"] == "training")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy().reset_index(drop=True)

    data["oof_probability"] = np.nan

    groups = data["patient_id"]
    y = data["referable_DR"].astype(int)

    group_cv = GroupKFold(n_splits=5)

    for fold, (train_idx, test_idx) in enumerate(
        group_cv.split(
            data[photometric_features],
            y,
            groups=groups
        ),
        start=1
    ):
        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )
            ),
        ])

        model.fit(
            data.loc[train_idx, photometric_features],
            y.iloc[train_idx]
        )

        data.loc[test_idx, "oof_probability"] = (
            model.predict_proba(
                data.loc[test_idx, photometric_features]
            )[:, 1]
        )

    assert data["oof_probability"].notna().all()

    # 只保留同一患者两只眼均在当前分辨率内
    complete = (
        data.groupby("patient_id")
        .filter(
            lambda g:
                len(g) == 2
                and g["eye_resolved"].nunique() == 2
        )
        .copy()
    )

    # 一只眼 referable，另一只眼 non-referable
    discordant = (
        complete.groupby("patient_id")
        .filter(
            lambda g:
                set(g["referable_DR"].astype(int)) == {0, 1}
        )
        .copy()
    )

    patient_ids = sorted(
        discordant["patient_id"].unique().tolist()
    )

    differences = []

    for patient_id in patient_ids:
        g = discordant[
            discordant["patient_id"] == patient_id
        ]

        positive = g[
            g["referable_DR"] == 1
        ].iloc[0]

        negative = g[
            g["referable_DR"] == 0
        ].iloc[0]

        difference = float(
            positive["oof_probability"]
            - negative["oof_probability"]
        )

        differences.append(difference)

        oof_detail_rows.append({
            "resolution": resolution,
            "patient_id": patient_id,
            "referable_eye": positive["eye_resolved"],
            "nonreferable_eye": negative["eye_resolved"],
            "referable_grade": int(
                positive["eye_DR_Level"]
            ),
            "nonreferable_grade": int(
                negative["eye_DR_Level"]
            ),
            "referable_probability": float(
                positive["oof_probability"]
            ),
            "nonreferable_probability": float(
                negative["oof_probability"]
            ),
            "probability_difference": difference,
            "correct_pair_ranking": bool(
                difference > 0
            ),
        })

    differences = np.asarray(differences)
    n_pairs = len(differences)
    correct = int((differences > 0).sum())

    if n_pairs:
        accuracy = correct / n_pairs

        p_value = binomtest(
            correct,
            n_pairs,
            p=0.5,
            alternative="greater"
        ).pvalue

        rng = np.random.default_rng(42)
        bootstrap_accuracy = []

        for _ in range(10000):
            sample = rng.choice(
                differences,
                size=n_pairs,
                replace=True
            )
            bootstrap_accuracy.append(
                np.mean(sample > 0)
            )

        ci_low, ci_high = np.quantile(
            bootstrap_accuracy,
            [0.025, 0.975]
        )

        mean_difference = float(
            differences.mean()
        )
    else:
        accuracy = np.nan
        p_value = np.nan
        ci_low = np.nan
        ci_high = np.nan
        mean_difference = np.nan

    oof_summary_rows.append({
        "source": "Nicheng",
        "resolution": resolution,
        "complete_two_eye_training_patients":
            complete["patient_id"].nunique(),
        "discordant_training_pairs": n_pairs,
        "correct_pair_rankings": correct,
        "pairwise_accuracy": accuracy,
        "bootstrap_CI_low": ci_low,
        "bootstrap_CI_high": ci_high,
        "exact_binomial_p": p_value,
        "mean_probability_difference":
            mean_difference,
    })

    print("\n" + "=" * 100)
    print("OOF RESOLUTION:", resolution)
    print(
        "Complete two-eye patients:",
        complete["patient_id"].nunique()
    )
    print("Discordant pairs:", n_pairs)
    print("Correct rankings:", correct)
    print(
        "Pairwise accuracy:",
        round(accuracy, 4) if n_pairs else "NA"
    )
    print(
        "95% bootstrap CI:",
        (
            round(ci_low, 4),
            round(ci_high, 4)
        ) if n_pairs else "NA"
    )
    print(
        "Exact binomial p:",
        round(p_value, 6) if n_pairs else "NA"
    )
    print(
        "Mean probability difference:",
        round(mean_difference, 4)
        if n_pairs else "NA"
    )


oof_pair_summary = pd.DataFrame(oof_summary_rows)
oof_pair_details = pd.DataFrame(oof_detail_rows)

summary_path = (
    output_dir /
    "training_OOF_discordant_fellow_eye_summary.csv"
)

details_path = (
    output_dir /
    "training_OOF_discordant_fellow_eye_details.csv"
)

oof_pair_summary.to_csv(summary_path, index=False)
oof_pair_details.to_csv(details_path, index=False)

print("\n" + "=" * 100)
print("FINAL OOF PAIRED SUMMARY")
print(oof_pair_summary.to_string(index=False))

print("\nSaved:")
print("-", summary_path.name)
print("-", details_path.name)


OOF RESOLUTION: 1736x1824
Complete two-eye patients: 175
Discordant pairs: 23
Correct rankings: 8
Pairwise accuracy: 0.3478
95% bootstrap CI: (np.float64(0.1739), np.float64(0.5652))
Exact binomial p: 0.95343
Mean probability difference: -0.0355

OOF RESOLUTION: 1976x1984
Complete two-eye patients: 97
Discordant pairs: 16
Correct rankings: 10
Pairwise accuracy: 0.625
95% bootstrap CI: (np.float64(0.375), np.float64(0.875))
Exact binomial p: 0.227249
Mean probability difference: 0.0272

FINAL OOF PAIRED SUMMARY
 source resolution  complete_two_eye_training_patients  discordant_training_pairs  correct_pair_rankings  pairwise_accuracy  bootstrap_CI_low  bootstrap_CI_high  exact_binomial_p  mean_probability_difference
Nicheng  1736x1824                                 175                         23                      8           0.347826          0.173913           0.565217          0.953430                    -0.035481
Nicheng  1976x1984                                  97             

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


photometric_features = [
    "gray_mean",
    "gray_std",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
]


def prepare_decomposition(df, split_name, resolution):
    x = df[
        (df["split"] == split_name)
        & (df["Source"] == "Nicheng")
        & (df["resolution"] == resolution)
    ].copy()

    # 只保留左右眼均在当前分辨率中的完整患者
    x = (
        x.groupby("patient_id")
        .filter(
            lambda g:
                len(g) == 2
                and g["eye_resolved"].nunique() == 2
        )
        .copy()
    )

    patient_means = (
        x.groupby("patient_id")[photometric_features]
        .transform("mean")
    )

    for feature in photometric_features:
        x[f"between_{feature}"] = patient_means[feature]
        x[f"within_{feature}"] = (
            x[feature] - patient_means[feature]
        )

    return x


feature_sets = {
    "full_eye_signal": photometric_features,

    "between_patient_signal": [
        f"between_{f}" for f in photometric_features
    ],

    "within_patient_signal": [
        f"within_{f}" for f in photometric_features
    ],
}


result_rows = []
pair_rows = []

for resolution in ["1736x1824", "1976x1984"]:

    train_dec = prepare_decomposition(
        fixed_eye_df,
        "training",
        resolution
    )

    val_dec = prepare_decomposition(
        fixed_eye_df,
        "validation",
        resolution
    )

    print("\n" + "=" * 100)
    print("RESOLUTION:", resolution)
    print(
        "Training patients:",
        train_dec["patient_id"].nunique()
    )
    print(
        "Validation patients:",
        val_dec["patient_id"].nunique()
    )
    print(
        "Training eyes:", len(train_dec),
        "| Validation eyes:", len(val_dec)
    )

    y_train = train_dec["referable_DR"].astype(int)
    y_val = val_dec["referable_DR"].astype(int)

    for feature_set_name, features in feature_sets.items():

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )
            ),
        ])

        model.fit(
            train_dec[features],
            y_train
        )

        probability = model.predict_proba(
            val_dec[features]
        )[:, 1]

        val_scored = val_dec[
            [
                "patient_id",
                "eye_resolved",
                "referable_DR",
                "eye_DR_Level"
            ]
        ].copy()

        val_scored["probability"] = probability

        auc = roc_auc_score(
            y_val,
            probability
        )

        ap = average_precision_score(
            y_val,
            probability
        )

        # 同患者左右眼不一致配对
        discordant = (
            val_scored.groupby("patient_id")
            .filter(
                lambda g:
                    set(g["referable_DR"].astype(int)) == {0, 1}
            )
        )

        differences = []

        for patient_id, group in discordant.groupby(
            "patient_id"
        ):
            positive = group[
                group["referable_DR"] == 1
            ].iloc[0]

            negative = group[
                group["referable_DR"] == 0
            ].iloc[0]

            differences.append(
                float(
                    positive["probability"]
                    - negative["probability"]
                )
            )

        differences = np.asarray(differences)

        if len(differences):
            tolerance = 1e-12

            pair_scores = np.where(
                differences > tolerance,
                1.0,
                np.where(
                    differences < -tolerance,
                    0.0,
                    0.5
                )
            )

            paired_concordance = float(
                pair_scores.mean()
            )

            mean_pair_difference = float(
                differences.mean()
            )
        else:
            paired_concordance = np.nan
            mean_pair_difference = np.nan

        result_rows.append({
            "source": "Nicheng",
            "resolution": resolution,
            "feature_set": feature_set_name,
            "training_patients":
                train_dec["patient_id"].nunique(),
            "validation_patients":
                val_dec["patient_id"].nunique(),
            "validation_eyes": len(val_dec),
            "roc_auc": float(auc),
            "average_precision": float(ap),
            "discordant_validation_pairs":
                len(differences),
            "paired_concordance":
                paired_concordance,
            "mean_paired_probability_difference":
                mean_pair_difference,
        })

        print(
            feature_set_name,
            "| AUC =", round(auc, 4),
            "| AP =", round(ap, 4),
            "| discordant pairs =", len(differences),
            "| paired concordance =",
            round(paired_concordance, 4)
            if len(differences) else "NA"
        )


signal_decomposition = pd.DataFrame(result_rows)

save_path = (
    output_dir /
    "between_within_patient_signal_decomposition.csv"
)

signal_decomposition.to_csv(
    save_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL SIGNAL DECOMPOSITION")
print(
    signal_decomposition
    .sort_values(
        ["resolution", "feature_set"]
    )
    .to_string(index=False)
)

print("\nSaved to:", save_path)


RESOLUTION: 1736x1824
Training patients: 175
Validation patients: 66
Training eyes: 350 | Validation eyes: 132
full_eye_signal | AUC = 0.7551 | AP = 0.5872 | discordant pairs = 6 | paired concordance = 0.6667
between_patient_signal | AUC = 0.7558 | AP = 0.578 | discordant pairs = 6 | paired concordance = 0.5
within_patient_signal | AUC = 0.4821 | AP = 0.3579 | discordant pairs = 6 | paired concordance = 0.3333

RESOLUTION: 1976x1984
Training patients: 97
Validation patients: 23
Training eyes: 194 | Validation eyes: 46
full_eye_signal | AUC = 0.5611 | AP = 0.849 | discordant pairs = 4 | paired concordance = 0.25
between_patient_signal | AUC = 0.6389 | AP = 0.8725 | discordant pairs = 4 | paired concordance = 0.5
within_patient_signal | AUC = 0.4778 | AP = 0.8227 | discordant pairs = 4 | paired concordance = 0.5

FINAL SIGNAL DECOMPOSITION
 source resolution            feature_set  training_patients  validation_patients  validation_eyes  roc_auc  average_precision  discordant_validation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


photometric_features = [
    "gray_mean",
    "gray_std",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
]

locality_rows = []

for resolution in ["1736x1824", "1976x1984"]:

    # --------------------------------------------------------
    # 1. 在官方训练集上训练同眼 DR 模型
    # --------------------------------------------------------
    train_sub = fixed_eye_df[
        (fixed_eye_df["split"] == "training")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    val_sub = fixed_eye_df[
        (fixed_eye_df["split"] == "validation")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42,
            )
        ),
    ])

    model.fit(
        train_sub[photometric_features],
        train_sub["referable_DR"].astype(int)
    )

    val_sub["probability"] = model.predict_proba(
        val_sub[photometric_features]
    )[:, 1]

    # --------------------------------------------------------
    # 2. 只保留双眼都具有相同固定分辨率的患者
    # --------------------------------------------------------
    complete = (
        val_sub.groupby("patient_id")
        .filter(
            lambda g:
                len(g) == 2
                and g["eye_resolved"].nunique() == 2
        )
        .copy()
    )

    evaluation_rows = []

    for patient_id, group in complete.groupby("patient_id"):

        group = group.reset_index(drop=True)

        first = group.iloc[0]
        second = group.iloc[1]

        evaluation_rows.append({
            "patient_id": patient_id,
            "eye_resolved": first["eye_resolved"],
            "probability": float(first["probability"]),
            "own_label": int(first["referable_DR"]),
            "fellow_label": int(second["referable_DR"]),
        })

        evaluation_rows.append({
            "patient_id": patient_id,
            "eye_resolved": second["eye_resolved"],
            "probability": float(second["probability"]),
            "own_label": int(second["referable_DR"]),
            "fellow_label": int(first["referable_DR"]),
        })

    evaluation = pd.DataFrame(evaluation_rows)

    own_auc = roc_auc_score(
        evaluation["own_label"],
        evaluation["probability"]
    )

    fellow_auc = roc_auc_score(
        evaluation["fellow_label"],
        evaluation["probability"]
    )

    auc_difference = own_auc - fellow_auc

    patient_labels = (
        complete.groupby("patient_id")["referable_DR"]
        .apply(lambda x: tuple(sorted(x.astype(int).tolist())))
    )

    bilateral_concordance = float(
        patient_labels.isin([(0, 0), (1, 1)]).mean()
    )

    # --------------------------------------------------------
    # 3. 患者级 bootstrap
    # --------------------------------------------------------
    rng = np.random.default_rng(42)
    patient_ids = evaluation["patient_id"].unique()

    bootstrap_own_auc = []
    bootstrap_fellow_auc = []
    bootstrap_difference = []

    for _ in range(10000):

        sampled_ids = rng.choice(
            patient_ids,
            size=len(patient_ids),
            replace=True
        )

        sampled_parts = []

        for bootstrap_index, patient_id in enumerate(sampled_ids):
            part = evaluation[
                evaluation["patient_id"] == patient_id
            ].copy()

            # 防止重复抽样患者被当成同一个索引组
            part["bootstrap_patient"] = bootstrap_index
            sampled_parts.append(part)

        sample = pd.concat(
            sampled_parts,
            ignore_index=True
        )

        if (
            sample["own_label"].nunique() < 2
            or sample["fellow_label"].nunique() < 2
        ):
            continue

        own_auc_b = roc_auc_score(
            sample["own_label"],
            sample["probability"]
        )

        fellow_auc_b = roc_auc_score(
            sample["fellow_label"],
            sample["probability"]
        )

        bootstrap_own_auc.append(own_auc_b)
        bootstrap_fellow_auc.append(fellow_auc_b)
        bootstrap_difference.append(
            own_auc_b - fellow_auc_b
        )

    own_ci = np.quantile(
        bootstrap_own_auc,
        [0.025, 0.975]
    )

    fellow_ci = np.quantile(
        bootstrap_fellow_auc,
        [0.025, 0.975]
    )

    difference_ci = np.quantile(
        bootstrap_difference,
        [0.025, 0.975]
    )

    locality_rows.append({
        "source": "Nicheng",
        "resolution": resolution,
        "validation_patients": len(patient_ids),
        "validation_eyes": len(evaluation),
        "bilateral_label_concordance":
            bilateral_concordance,
        "own_eye_auc": own_auc,
        "own_eye_CI_low": own_ci[0],
        "own_eye_CI_high": own_ci[1],
        "fellow_eye_auc": fellow_auc,
        "fellow_eye_CI_low": fellow_ci[0],
        "fellow_eye_CI_high": fellow_ci[1],
        "own_minus_fellow_auc": auc_difference,
        "difference_CI_low": difference_ci[0],
        "difference_CI_high": difference_ci[1],
    })

    print("\n" + "=" * 100)
    print("RESOLUTION:", resolution)
    print("Validation patients:", len(patient_ids))
    print("Bilateral label concordance:",
          round(bilateral_concordance, 4))
    print(
        "Own-eye AUC:",
        round(own_auc, 4),
        "| 95% CI:",
        tuple(np.round(own_ci, 4))
    )
    print(
        "Fellow-eye AUC:",
        round(fellow_auc, 4),
        "| 95% CI:",
        tuple(np.round(fellow_ci, 4))
    )
    print(
        "Own minus fellow AUC:",
        round(auc_difference, 4),
        "| 95% CI:",
        tuple(np.round(difference_ci, 4))
    )


locality_summary = pd.DataFrame(locality_rows)

save_path = (
    output_dir /
    "own_eye_vs_fellow_eye_locality_test.csv"
)

locality_summary.to_csv(
    save_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL LOCALITY SUMMARY")
print(locality_summary.to_string(index=False))

print("\nSaved to:", save_path)


RESOLUTION: 1736x1824
Validation patients: 66
Bilateral label concordance: 0.9091
Own-eye AUC: 0.7556 | 95% CI: (np.float64(0.627), np.float64(0.8734))
Fellow-eye AUC: 0.7535 | 95% CI: (np.float64(0.6248), np.float64(0.8718))
Own minus fellow AUC: 0.002 | 95% CI: (np.float64(-0.0082), np.float64(0.0147))

RESOLUTION: 1976x1984
Validation patients: 23
Bilateral label concordance: 0.8261
Own-eye AUC: 0.5611 | 95% CI: (np.float64(0.3075), np.float64(0.8282))
Fellow-eye AUC: 0.6167 | 95% CI: (np.float64(0.3591), np.float64(0.8914))
Own minus fellow AUC: -0.0556 | 95% CI: (np.float64(-0.1759), np.float64(0.0263))

FINAL LOCALITY SUMMARY
 source resolution  validation_patients  validation_eyes  bilateral_label_concordance  own_eye_auc  own_eye_CI_low  own_eye_CI_high  fellow_eye_auc  fellow_eye_CI_low  fellow_eye_CI_high  own_minus_fellow_auc  difference_CI_low  difference_CI_high
Nicheng  1736x1824                   66              132                     0.909091     0.755561        0.626

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from scipy.stats import binomtest


photometric_features = [
    "gray_mean",
    "gray_std",
    "red_mean",
    "green_mean",
    "blue_mean",
    "saturation_mean",
    "value_mean",
]

ordinal_rows = []
ordinal_summary_rows = []

for resolution in ["1736x1824", "1976x1984"]:

    # ========================================================
    # 1. 用训练集学习连续的 DR 严重度分数
    # ========================================================

    train_sub = fixed_eye_df[
        (fixed_eye_df["split"] == "training")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    val_sub = fixed_eye_df[
        (fixed_eye_df["split"] == "validation")
        & (fixed_eye_df["Source"] == "Nicheng")
        & (fixed_eye_df["resolution"] == resolution)
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "regressor",
            Ridge(alpha=1.0)
        ),
    ])

    model.fit(
        train_sub[photometric_features],
        train_sub["eye_DR_Level"].astype(float)
    )

    val_sub["predicted_grade_score"] = model.predict(
        val_sub[photometric_features]
    )

    # ========================================================
    # 2. 只保留双眼都位于当前固定分辨率中的患者
    # ========================================================

    complete = (
        val_sub.groupby("patient_id")
        .filter(
            lambda g:
                len(g) == 2
                and g["eye_resolved"].nunique() == 2
        )
        .copy()
    )

    unequal = (
        complete.groupby("patient_id")
        .filter(
            lambda g:
                g["eye_DR_Level"].nunique() == 2
        )
        .copy()
    )

    patient_ids = sorted(
        unequal["patient_id"].unique().tolist()
    )

    score_differences = []
    grade_differences = []

    for patient_id in patient_ids:

        group = unequal[
            unequal["patient_id"] == patient_id
        ].copy()

        higher = group.sort_values(
            "eye_DR_Level",
            ascending=False
        ).iloc[0]

        lower = group.sort_values(
            "eye_DR_Level",
            ascending=True
        ).iloc[0]

        score_difference = float(
            higher["predicted_grade_score"]
            - lower["predicted_grade_score"]
        )

        grade_difference = int(
            higher["eye_DR_Level"]
            - lower["eye_DR_Level"]
        )

        score_differences.append(score_difference)
        grade_differences.append(grade_difference)

        ordinal_rows.append({
            "resolution": resolution,
            "patient_id": patient_id,
            "higher_grade_eye": higher["eye_resolved"],
            "lower_grade_eye": lower["eye_resolved"],
            "higher_grade": int(higher["eye_DR_Level"]),
            "lower_grade": int(lower["eye_DR_Level"]),
            "true_grade_difference": grade_difference,
            "higher_eye_predicted_score": float(
                higher["predicted_grade_score"]
            ),
            "lower_eye_predicted_score": float(
                lower["predicted_grade_score"]
            ),
            "predicted_score_difference":
                score_difference,
            "correct_severity_ranking":
                bool(score_difference > 0),
        })

    score_differences = np.asarray(
        score_differences,
        dtype=float
    )

    grade_differences = np.asarray(
        grade_differences,
        dtype=float
    )

    n_pairs = len(score_differences)
    correct = int((score_differences > 0).sum())

    if n_pairs:
        accuracy = correct / n_pairs

        exact_p = binomtest(
            correct,
            n_pairs,
            p=0.5,
            alternative="greater"
        ).pvalue

        rng = np.random.default_rng(42)
        bootstrap_accuracy = []
        bootstrap_mean_difference = []

        for _ in range(10000):
            indices = rng.integers(
                0,
                n_pairs,
                size=n_pairs
            )

            sample_scores = score_differences[indices]

            bootstrap_accuracy.append(
                np.mean(sample_scores > 0)
            )

            bootstrap_mean_difference.append(
                np.mean(sample_scores)
            )

        accuracy_ci = np.quantile(
            bootstrap_accuracy,
            [0.025, 0.975]
        )

        mean_difference_ci = np.quantile(
            bootstrap_mean_difference,
            [0.025, 0.975]
        )

        mean_score_difference = float(
            score_differences.mean()
        )

        mean_true_grade_difference = float(
            grade_differences.mean()
        )
    else:
        accuracy = np.nan
        exact_p = np.nan
        accuracy_ci = [np.nan, np.nan]
        mean_difference_ci = [np.nan, np.nan]
        mean_score_difference = np.nan
        mean_true_grade_difference = np.nan

    ordinal_summary_rows.append({
        "source": "Nicheng",
        "resolution": resolution,
        "complete_two_eye_validation_patients":
            complete["patient_id"].nunique(),
        "unequal_grade_validation_pairs": n_pairs,
        "correct_severity_rankings": correct,
        "severity_pair_accuracy": accuracy,
        "accuracy_CI_low": accuracy_ci[0],
        "accuracy_CI_high": accuracy_ci[1],
        "exact_binomial_p": exact_p,
        "mean_true_grade_difference":
            mean_true_grade_difference,
        "mean_predicted_score_difference":
            mean_score_difference,
        "score_difference_CI_low":
            mean_difference_ci[0],
        "score_difference_CI_high":
            mean_difference_ci[1],
    })

    print("\n" + "=" * 100)
    print("RESOLUTION:", resolution)
    print(
        "Complete two-eye validation patients:",
        complete["patient_id"].nunique()
    )
    print("Unequal-grade pairs:", n_pairs)
    print("Correct severity rankings:", correct)
    print(
        "Pairwise accuracy:",
        round(accuracy, 4) if n_pairs else "NA"
    )
    print(
        "95% bootstrap CI:",
        tuple(np.round(accuracy_ci, 4))
        if n_pairs else "NA"
    )
    print(
        "Exact binomial p:",
        round(exact_p, 6) if n_pairs else "NA"
    )
    print(
        "Mean predicted score difference:",
        round(mean_score_difference, 4)
        if n_pairs else "NA"
    )


ordinal_pair_summary = pd.DataFrame(
    ordinal_summary_rows
)

ordinal_pair_details = pd.DataFrame(
    ordinal_rows
)

summary_path = (
    output_dir /
    "ordinal_DR_fellow_eye_locality_summary.csv"
)

details_path = (
    output_dir /
    "ordinal_DR_fellow_eye_locality_details.csv"
)

ordinal_pair_summary.to_csv(
    summary_path,
    index=False
)

ordinal_pair_details.to_csv(
    details_path,
    index=False
)

print("\n" + "=" * 100)
print("FINAL ORDINAL LOCALITY SUMMARY")
print(ordinal_pair_summary.to_string(index=False))

print("\nSaved:")
print("-", summary_path.name)
print("-", details_path.name)


RESOLUTION: 1736x1824
Complete two-eye validation patients: 66
Unequal-grade pairs: 31
Correct severity rankings: 15
Pairwise accuracy: 0.4839
95% bootstrap CI: (np.float64(0.3226), np.float64(0.6452))
Exact binomial p: 0.63995
Mean predicted score difference: 0.0343

RESOLUTION: 1976x1984
Complete two-eye validation patients: 23
Unequal-grade pairs: 11
Correct severity rankings: 4
Pairwise accuracy: 0.3636
95% bootstrap CI: (np.float64(0.0909), np.float64(0.6364))
Exact binomial p: 0.886719
Mean predicted score difference: -0.0601

FINAL ORDINAL LOCALITY SUMMARY
 source resolution  complete_two_eye_validation_patients  unequal_grade_validation_pairs  correct_severity_rankings  severity_pair_accuracy  accuracy_CI_low  accuracy_CI_high  exact_binomial_p  mean_true_grade_difference  mean_predicted_score_difference  score_difference_CI_low  score_difference_CI_high
Nicheng  1736x1824                                    66                              31                         15         

In [ ]:
from pathlib import Path
import json

conclusion_path = output_dir / "DeepDRiD_Smoke_Test_Conclusion.md"
summary_path = output_dir / "audit_summary.json"

conclusion_text = """# DeepDRiD Smoke Test Conclusion

## Dataset and audit status

- Dataset: DeepDRiD v1.1 regular fundus subset
- Training: 1,200 images, 300 patients
- Validation: 400 images, 100 patients
- Four images per patient, two images per eye
- No patient or image-ID overlap between official training and validation sets
- All 1,600 images linked successfully to label and source metadata
- Eleven filename/label eye-identity conflicts were identified in four training patients:
  56, 77, 164, and 167
- Eye identity was resolved from the non-null left/right eye DR label while preserving
  original image IDs and filenames

## Main empirical findings

### 1. Simple global metrics produced apparently strong diagnostic performance

Using brightness, colour, border, sharpness, image dimensions, and file-level properties:

- Official image-quality prediction reached ROC AUC 0.780
- Referable-DR prediction reached ROC AUC 0.812

Thus, high diagnostic prediction was achievable without explicit lesion analysis.

### 2. Acquisition resolution was strongly associated with DR prevalence

Within the dominant Nicheng source:

- 1736x1824 images had referable-DR prevalence of approximately 28–34%
- 1976x1984 images had referable-DR prevalence of approximately 71–78%

This association occurred in both official training and validation partitions,
indicating a stable acquisition shortcut.

### 3. Fixing source and resolution reduced, but did not eliminate, prediction

Within Nicheng and fixed resolution:

- 1736x1824 photometric ROC AUC: 0.762
- 1976x1984 photometric ROC AUC: 0.561

The high aggregate AUC was therefore partly driven by acquisition structure.

### 4. Remaining signal was predominantly between patients, not between eyes

For 1736x1824 images:

- Between-patient photometric AUC: 0.756
- Within-patient photometric AUC: 0.482

For 1976x1984 images:

- Between-patient photometric AUC: 0.639
- Within-patient photometric AUC: 0.478

Removing each patient's bilateral mean eliminated nearly all discriminative signal.

### 5. Own-eye prediction was not better than fellow-eye prediction

For 1736x1824 images:

- Own-eye AUC: 0.756
- Fellow-eye AUC: 0.754
- Difference: 0.002
- 95% bootstrap CI for the difference: -0.008 to 0.015

The same eye's image was therefore no more informative about its own binary DR label
than about the fellow eye's label.

### 6. Eye-specific severity ranking was unsuccessful

Among validation patients with unequal left/right DR grades:

- 1736x1824: 15/31 correct, accuracy 0.484
- 1976x1984: 4/11 correct, accuracy 0.364

Neither result showed evidence of above-chance eye-specific severity ranking.

## Interpretation

DeepDRiD demonstrates a separation between predictive performance and local
diagnostic observability.

Simple global image statistics can predict DR labels at apparently useful AUC,
yet that predictive signal:

1. is partly encoded by acquisition resolution,
2. is concentrated primarily between patients,
3. predicts the fellow eye nearly as well as the imaged eye, and
4. does not reliably rank the more severely affected eye within the same patient.

Therefore, conventional validation AUC alone does not establish that a predictor
uses eye-local disease evidence.

## Scope and limitations

This smoke test does not show that:

- retinal lesions are absent from the images,
- deep neural networks cannot use local pathology,
- all DeepDRiD performance is spurious, or
- the same mechanism necessarily occurs in other modalities.

The current result is a dataset-specific mechanism demonstration using simple
global features. Cross-dataset, representation-level, and cross-modal validation
remain necessary.

## Dataset role in the project

DeepDRiD should be treated as:

**Mechanism validation for the distinction between diagnostic predictability and
eye-local diagnostic observability.**

It should not be treated as the sole primary dataset for establishing a general
cross-modal law.
"""

conclusion_path.write_text(
    conclusion_text,
    encoding="utf-8"
)

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

summary["smoke_test_status"] = "Completed"
summary["dataset_role"] = (
    "Mechanism validation for the distinction between diagnostic "
    "predictability and eye-local diagnostic observability."
)

summary["key_findings"] = {
    "simple_metric_referable_DR_auc": 0.812449,
    "dominant_acquisition_shortcut": (
        "Image resolution was strongly associated with DR prevalence."
    ),
    "fixed_1736_photometric_auc": 0.761602,
    "fixed_1976_photometric_auc": 0.561111,
    "within_patient_auc_1736": 0.482053,
    "within_patient_auc_1976": 0.477778,
    "own_eye_auc_1736": 0.755561,
    "fellow_eye_auc_1736": 0.753539,
    "own_minus_fellow_auc_1736": 0.002022,
    "ordinal_pair_accuracy_1736": 0.483871,
    "ordinal_pair_accuracy_1976": 0.363636
}

summary["interpretation"] = (
    "Apparently strong diagnostic prediction was driven primarily by "
    "acquisition and between-patient signals, with little evidence of "
    "eye-specific locality from the tested global image metrics."
)

summary["limitations"] = [
    "The analysis used simple global image metrics rather than deep representations.",
    "The result is specific to DeepDRiD and does not establish a cross-modal law.",
    "High bilateral DR concordance limits the number of informative discordant pairs.",
    "The findings do not imply that retinal lesions or local pathological signals are absent."
]

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Conclusion saved:", conclusion_path)
print("Audit summary updated:", summary_path)
print("\nCurrent files:")
for path in sorted(output_dir.iterdir()):
    print("-", path.name)

Conclusion saved: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/DeepDRiD/Smoke_Test_v0.1/20260718T220050Z/DeepDRiD_Smoke_Test_Conclusion.md
Audit summary updated: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/DeepDRiD/Smoke_Test_v0.1/20260718T220050Z/audit_summary.json

Current files:
- DeepDRiD_Smoke_Test_Conclusion.md
- audit_summary.json
- between_within_patient_signal_decomposition.csv
- deepdrid_canonical_metadata.csv
- deepdrid_image_profile.csv
- discordant_fellow_eye_pair_details.csv
- discordant_fellow_eye_pair_summary.csv
- eye_level_quality_source_confounding_audit.csv
- eye_name_conflicts.csv
- fixed_resolution_DR_shortcut_audit.csv
- ordinal_DR_fellow_eye_locality_details.csv
- ordinal_DR_fellow_eye_locality_summary.csv
- own_eye_vs_fellow_eye_locality_test.csv
- simple_quality_baseline_results.csv
- simple_quality_validation_predictions.csv
- single_metric_validation_auc.csv
- training_OOF_discordant_fellow_eye_d

In [3]:
from pathlib import Path
from textwrap import dedent

out_dir = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "06_Data_Records/Retinal_DR/Smoke_Test_v0.1"
)
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "Retinal_DR_Smoke_Test_v0.1_Results_and_Decision.md"

text = dedent("""
# Retinal DR Smoke Test v0.1 — Results and Decision

**Date:** 2026-07-19
**Status:** Initial dataset and mechanism smoke test completed
**Decision:** Proceed to the frozen-representation baseline under explicit confounding and locality controls.

## 1. Purpose

This smoke test evaluated whether public retinal diabetic-retinopathy datasets can support a patient-safe and acquisition-aware investigation of diagnostic observability.

The immediate objective was not to maximise diagnostic performance. It was to determine whether apparently strong prediction can be distinguished from:

1. acquisition and image-quality shortcuts;
2. patient-level disease or identity information;
3. genuinely eye-local diagnostic evidence.

## 2. Datasets audited

### IDRiD

The official Disease Grading subset contained:

- 413 training images;
- 103 testing images;
- 516 images in total;
- official DR grading CSV files;
- uniform image dimensions of 4288 × 2848 pixels.

No exact duplicate images were detected in the training subset. Measurable image-quality variation was present.

Patient-level identifiers could not be reliably recovered from the released grading data. IDRiD is therefore retained for acquisition-profile development, perturbation design and mechanism validation, but not as the principal patient-independent or blind-target dataset.

### DeepDRiD

The audited regular-fundus subsets contained:

- 1,200 training images from 300 patients;
- 400 validation images from 100 patients;
- four images per patient;
- two images per eye;
- no patient or image overlap between training and validation.

Eleven filename-versus-eye-label conflicts were identified in four training patients. Eye identity was resolved using the official non-null left/right DR label fields, and all conflicts were recorded.

The archive license was verified as **CC BY-SA 4.0**.

## 3. Main quantitative findings

### 3.1 Apparently strong simple prediction

Using simple global quality and acquisition-related measurements:

- official quality plus acquisition variables achieved validation AUC ≈ 0.780;
- referable-DR plus acquisition-related measurements achieved validation AUC ≈ 0.812.

These values initially suggested substantial diagnostic information in simple image statistics.

### 3.2 Acquisition and resolution shortcut

DeepDRiD contained a strong relationship between image resolution, acquisition source and DR prevalence.

The two dominant resolutions showed markedly different referable-DR prevalence. Resolution and acquisition variables therefore acted as important diagnostic shortcuts.

Within fixed-resolution subsets:

- 1736 × 1824 photometric-feature AUC ≈ 0.762;
- 1976 × 1984 photometric-feature AUC ≈ 0.561.

Thus, resolution adjustment reduced but did not completely remove apparently diagnostic signal.

### 3.3 Between-patient versus within-patient information

For the 1736 × 1824 subset:

- between-patient AUC ≈ 0.756;
- within-patient AUC ≈ 0.482.

For the 1976 × 1984 subset:

- between-patient AUC ≈ 0.639;
- within-patient AUC ≈ 0.478.

The remaining simple-metric signal was therefore predominantly between patients rather than within patients.

### 3.4 Own-eye versus fellow-eye prediction

For the 1736 × 1824 subset:

- own-eye AUC ≈ 0.756;
- fellow-eye AUC ≈ 0.754;
- AUC difference ≈ 0.002;
- bootstrap confidence interval approximately −0.008 to 0.015;
- bilateral label concordance ≈ 0.909.

For the 1976 × 1984 subset:

- own-eye AUC ≈ 0.561;
- fellow-eye AUC ≈ 0.617.

Simple global features did not provide stable evidence that the target eye contained more diagnostic information than the fellow eye.

### 3.5 Within-patient ordinal severity ranking

Among fellow-eye pairs with unequal DR grades:

- 1736 × 1824 ranking accuracy: 15/31 ≈ 0.484;
- 1976 × 1984 ranking accuracy: 4/11 ≈ 0.364.

The simple features could not reliably identify which eye had the more severe DR grade.

## 4. Interpretation

The DeepDRiD smoke test demonstrates a dataset-specific distinction between:

- **diagnostic predictability**, and
- **eye-local diagnostic observability**.

Simple global image measurements can produce apparently strong diagnostic AUC while relying heavily on acquisition structure, resolution, patient-level disease state and bilateral concordance.

High image-level diagnostic performance is therefore not sufficient evidence that a representation contains eye-specific pathological information.

## 5. What has not been established

The current findings do not show that:

- all deep-learning models rely only on shortcuts;
- lesion-level information is absent from fundus images;
- the same mechanism necessarily holds in every retinal dataset;
- the result is already a universal cross-modal law.

The evidence currently supports a carefully bounded mechanism result within the audited DeepDRiD setting.

## 6. Dataset roles after the smoke test

### IDRiD

Role:

> Acquisition-profile development, perturbation design and mechanism validation; not a primary patient-independent source or sealed blind target.

### DeepDRiD

Role:

> Mechanism validation for the distinction between diagnostic predictability and eye-local diagnostic observability; not a sealed blind target after label-informed development.

## 7. Decision

**GO, with constraints.**

Proceed to the frozen-representation baseline, but require:

1. patient-grouped training and evaluation;
2. source- and resolution-stratified analysis;
3. explicit own-eye versus fellow-eye comparison;
4. between-patient and within-patient decomposition;
5. no use of unstratified image-level AUC alone as evidence of diagnostic observability;
6. replication on an independent dataset before making any general claim.

## 8. Next experimental gate

The next question is:

> Can a frozen general-purpose or biomedical image representation recover reproducible eye-local diagnostic information beyond simple quality, acquisition and patient-level baselines?

Step 6 should compare frozen embeddings against:

- simple image-quality baselines;
- acquisition/source baselines;
- patient-level and fellow-eye baselines;
- fixed-resolution and fixed-source subsets;
- within-patient severity and eye-locality tests.

Only representations that add stable own-eye or within-patient information should advance to the prototype kill test.
""").strip() + "\n"

out_file.write_text(text, encoding="utf-8")

print("Saved:")
print(out_file)
print("Size:", out_file.stat().st_size, "bytes")

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Smoke_Test_v0.1/Retinal_DR_Smoke_Test_v0.1_Results_and_Decision.md
Size: 6377 bytes


In [4]:
from pathlib import Path
import nbformat as nbf

notebook_path = Path(
    "/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/"
    "02_Dataset_Map/Retinal_DR_Frozen_Representation_Baseline_v0.1.ipynb"
)

if notebook_path.exists():
    print("Already exists:")
    print(notebook_path)
else:
    nb = nbf.v4.new_notebook()

    nb["cells"] = [
        nbf.v4.new_markdown_cell(
            "# Retinal DR Frozen Representation Baseline v0.1\n\n"
            "**Step 6 of the Retinal DR smoke-test plan**\n\n"
            "Purpose: test whether frozen image representations recover "
            "eye-local diagnostic information beyond acquisition, quality, "
            "resolution and patient-level baselines."
        ),

        nbf.v4.new_markdown_cell(
            "## 00. Runtime, Drive mount and configuration"
        ),

        nbf.v4.new_code_cell(
            "from google.colab import drive\n"
            "drive.mount('/content/drive')"
        ),

        nbf.v4.new_code_cell(
            "from pathlib import Path\n\n"
            "PROJECT_ROOT = Path(\n"
            "    '/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability'\n"
            ")\n\n"
            "DEEPDRID_ROOT = (\n"
            "    PROJECT_ROOT / '02_Dataset_Map' /\n"
            "    'DeepDRiD_Official_Raw' / 'DeepDRiD_v1.1_Extracted'\n"
            ")\n\n"
            "OUTPUT_ROOT = (\n"
            "    PROJECT_ROOT / '06_Data_Records' / 'Retinal_DR' /\n"
            "    'Frozen_Representation_Baseline_v0.1'\n"
            ")\n"
            "OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n\n"
            "print('DeepDRiD root:', DEEPDRID_ROOT)\n"
            "print('Output root:', OUTPUT_ROOT)"
        ),

        nbf.v4.new_markdown_cell(
            "## 01. Load frozen metadata and define patient-safe evaluation"
        ),

        nbf.v4.new_markdown_cell(
            "## 02. Frozen encoder and embedding extraction"
        ),

        nbf.v4.new_markdown_cell(
            "## 03. Patient-grouped diagnostic baseline"
        ),

        nbf.v4.new_markdown_cell(
            "## 04. Source- and resolution-stratified evaluation"
        ),

        nbf.v4.new_markdown_cell(
            "## 05. Between-patient versus within-patient decomposition"
        ),

        nbf.v4.new_markdown_cell(
            "## 06. Own-eye versus fellow-eye locality test"
        ),

        nbf.v4.new_markdown_cell(
            "## 07. Within-patient ordinal severity ranking"
        ),

        nbf.v4.new_markdown_cell(
            "## 08. Results, decision gate and reproducibility outputs"
        ),
    ]

    nb["metadata"] = {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "language_info": {"name": "python"},
    }

    nbf.write(nb, notebook_path)

    print("Created:")
    print(notebook_path)

Created:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/Retinal_DR_Frozen_Representation_Baseline_v0.1.ipynb
